# 02. Tratamento e Consolidação de Dados ETL (SmartQuestion)

Este notebook é responsável pelo processamento de dados dos relatórios do SmartQuestion e alimentação das tabelas do banco de dados.

In [ ]:
# 1. Setup do Ambiente e Importações Centralizadas
from __future__ import annotations

import os
import sys
import glob
import time
import json
import yaml
import hashlib
import warnings
import calendar
import traceback
from datetime import datetime, date, timezone, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
from dotenv import load_dotenv
from supabase import create_client
from IPython.display import HTML, Markdown, display

try:
    from unidecode import unidecode
except ImportError:
    unidecode = lambda x: x

# Filtrar avisos de depreciação do Pandas (FutureWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Localizar a raiz do projeto dinamicamente
caminho_atual = Path.cwd().resolve()
for candidato in [caminho_atual, *caminho_atual.parents]:
    if (candidato / "SCRIPTS").is_dir() and (candidato / "DB").is_dir():
        raiz_projeto = candidato
        break
else:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")

for p in [raiz_projeto, raiz_projeto / "SCRIPTS", raiz_projeto / "SCRIPTS" / "FUNCTIONS"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from FUNCTIONS.function import (
    aplicar_estilo_listrado_xlsx,
    aplicar_formatacao_excel,
    buscar_arquivo_mais_recente,
    carregar_config_referencia,
    carregar_env,
    consultar_tabela_supabase,
    converter_data_excel,
    converter_numero_br,
    detectar_raiz_projeto,
    dividir_seguro,
    exportar_varias_abas_xlsx,
    exportar_xlsx_formatado,
    extrair_data_nome_arquivo,
    garantir_colunas,
    ler_aba_excel_flex,
    normalizar_texto,
    obter_cliente_supabase,
    renomear_colunas_existentes,
)

# Carregar configurações do projeto (config.yaml)
config_ref = carregar_config_referencia(raiz_projeto)

caminho_config_yaml = raiz_projeto / "SCRIPTS" / "CONFIG" / "config.yaml"
if caminho_config_yaml.exists():
    with open(caminho_config_yaml, "r", encoding="utf-8") as f:
        config_yaml = yaml.safe_load(f)
else:
    raise FileNotFoundError(f"Arquivo de configuração não encontrado em: {caminho_config_yaml}")

# Leitura estrita de diretórios, tabelas e datas de referência (Lança KeyError se faltar chave)
DIR_BD_SQ = Path(config_yaml["caminhos"]["bd_smartquestion"])

TAB_VINCULOS_STAGING = config_yaml["supabase"]["tabelas"]["vinculos_staging"]
TAB_VISITAS_STAGING = config_yaml["supabase"]["tabelas"]["visitas_staging"]
TAB_VISITAS_FATO = config_yaml["supabase"]["tabelas"]["visitas_fato"]
TAB_INATIVACAO_PRODUTORES = config_yaml["supabase"]["tabelas"]["inativacao_produtores_staging"]
TAB_INATIVACAO_CONSULTORES = config_yaml["supabase"]["tabelas"]["inativacao_consultores_staging"]
TAB_CONSISTENCIA_FATO = config_yaml["supabase"]["tabelas"]["consistencia_fato"]
TAB_MOVIMENTACAO_FATO = config_yaml["supabase"]["tabelas"]["movimentacao_fato"]

# Datas de corte centralizadas
DATA_INICIAL_ANALISE = config_yaml["referencia"]["data_inicial_analise"]
DATA_INICIAL_ELABORE = config_yaml["referencia"]["data_inicial_elabore"]
DATA_INICIAL_FATO_VISITAS = config_yaml["referencia"]["data_inicial_fato_visitas"]
PERIODO_CHECAGEM_INICIO = config_yaml["referencia"]["periodo_checagem_inicio"]

DIRETORIO_ENV = str(raiz_projeto / "SCRIPTS")
NOTEBOOK_DIR = str(raiz_projeto / "SCRIPTS")

# Carregar variáveis de ambiente (.env)
caminho_env = raiz_projeto / "SCRIPTS" / "CONFIG" / ".env"
if caminho_env.exists():
    load_dotenv(caminho_env)
else:
    load_dotenv()

# Inicializar cliente Supabase globalmente
supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_SERVICE_KEY")
if supabase_url and supabase_key:
    supabase = create_client(supabase_url, supabase_key)
else:
    supabase = None

print(f"Raiz do projeto: {raiz_projeto}")
print(f"Configuração carregada: {config_ref}")
print(f"Diretório BD_SMARTQUESTION: {DIR_BD_SQ}")
print(f"Datas de Corte: Análise={DATA_INICIAL_ANALISE} | Elabore={DATA_INICIAL_ELABORE} | Fato Visitas={DATA_INICIAL_FATO_VISITAS}")

# 2. Inativação de Produtores


In [ ]:
def etl_inativacao(diretorio=None, DIRETORIO_ENV=None):
    """
    Função completa para ETL diário de dados de inativação:
    1. Importa o arquivo Excel mais recente com _LISTA_INATIVACAO.xlsx
    2. Filtra registros novos
    3. Insere no Supabase

    Args:
        diretorio: Diretório onde procurar o arquivo Excel (opcional)

    Returns:
        bool: True se o processo foi concluído com sucesso, False caso contrário
    """

    try:
        print("=== ETL DIÁRIO DE INATIVAÇÃO ===")
        inicio_total = datetime.now()

        # ETAPA 1: IMPORTAR ARQUIVO EXCEL
        print("\n🔍 ETAPA 1: IMPORTANDO ARQUIVO EXCEL")

        # Salvar diretório atual

        # Mudar para o diretório especificado se fornecido

        # Buscar qualquer arquivo com o padrão _LISTA_INATIVACAO.xlsx
        diretorio_alvo = Path(diretorio) if diretorio else DIR_BD_SQ
        print(f"Diretório dos arquivos: {diretorio_alvo}")
        padrao = str(diretorio_alvo / "*_LISTA_INATIVACAO.xlsx")
        arquivos = glob.glob(padrao)

        if not arquivos:
            print(f"❌ Nenhum arquivo encontrado com o padrão {padrao}")
            # Voltar ao diretório original
            return False

        # Ordenar arquivos por data de modificação (mais recente primeiro)
        arquivos_ordenados = sorted(arquivos, key=os.path.getmtime, reverse=True)
        arquivo_mais_recente = arquivos_ordenados[0]

        print(f"✅ Arquivo mais recente encontrado: {arquivo_mais_recente}")

        try:
            # Definir colunas para importação
            colunas_inativacao = [
                'Número do atendimento:', 'Consultor(a):', 'Projeto',
                'Código do(a) produtor(a):', 'Produtor(a):', 'Propriedade:',
                'Grupo Ponto Atendimento', 'Data da solicitação:',
                'Data da inativação:', 'Motivo da inativação:',
                'Se outro, qual motivo?', 'Produtor(a) ativo(a)?'
            ]

            # Importar o arquivo Excel
            df_inativacao = pd.read_excel(arquivo_mais_recente, header=1, usecols=colunas_inativacao)

            # Mapear nomes de colunas
            inativacao_mapping = {
                'Número do atendimento:': 'id_atendimento',
                'Consultor(a):': 'nome_consultor',
                'Projeto': 'projeto',
                'Código do(a) produtor(a):': 'codigo_lr',
                'Produtor(a):': 'nome_produtor',
                'Propriedade:': 'nome_propriedade',
                'Grupo Ponto Atendimento': 'grupo_ponto_atendimento',
                'Data da solicitação:': 'data_solicitacao',
                'Data da inativação:': 'data_inativacao',
                'Motivo da inativação:': 'motivo_inativacao',
                'Se outro, qual motivo?': 'outro_motivo',
                'Produtor(a) ativo(a)?': 'produtor_ativo'
            }

            # Renomear colunas
            df_inativacao.rename(columns=inativacao_mapping, inplace=True)

            print(f"✅ Arquivo importado com sucesso: {arquivo_mais_recente} ({len(df_inativacao)} registros)")

        except Exception as e:
            print(f"❌ Erro ao importar {arquivo_mais_recente}: {str(e)}")
            # Voltar ao diretório original
            return False

        # Voltar ao diretório original

        if len(df_inativacao) == 0:
            print("❌ Nenhum dado encontrado no arquivo.")
            return False

        # ETAPA 2: BUSCAR REGISTROS NOVOS
        print("\n🔍 ETAPA 2: FILTRANDO REGISTROS NOVOS")
        
        # Carregar credenciais do Supabase
        key = os.getenv('SUPABASE_SERVICE_KEY')
        url = os.getenv('SUPABASE_URL')
        
        if not key or not url:
            print("❌ Credenciais não encontradas!")
            return False
        
        # Inicializar cliente Supabase
        supabase = create_client(url, key)
        
        # Buscar todos os id_atendimento já existentes no Supabase
        print("🔍 Buscando registros existentes no Supabase...")
        
        try:
            resultado = supabase.table(TAB_INATIVACAO_PRODUTORES) \
                .select("id_atendimento") \
                .execute()
        
            if resultado.data:
                ids_existentes = set(r['id_atendimento'] for r in resultado.data)
                print(f"✅ {len(ids_existentes)} registros já existem no Supabase")
            else:
                ids_existentes = set()
                print("⚠️ Nenhum registro encontrado no Supabase. Todos serão inseridos.")
        
        except Exception as e:
            print(f"⚠️ Erro ao buscar registros existentes: {str(e)}")
            ids_existentes = set()
        
        # Converter data_solicitacao para datetime
        df_inativacao['data_solicitacao'] = pd.to_datetime(df_inativacao['data_solicitacao'], errors='coerce')
        
        # Filtrar apenas registros cujo id_atendimento NÃO existe no Supabase
        df_novos = df_inativacao[~df_inativacao['id_atendimento'].isin(ids_existentes)].copy()
        
        if len(df_novos) == 0:
            print("✅ Nenhum registro novo encontrado.")
            return True
        
        print(f"✅ Encontrados {len(df_novos)} registros novos para inserir.")
        # ETAPA 3: PROCESSAR E INSERIR NO SUPABASE
        print("\n🔄 ETAPA 3: PROCESSANDO E INSERINDO DADOS")

        # Ajustar tipos de dados
        print("🔄 Ajustando tipos de dados...")

        # Converter colunas de data para datetime
        if 'data_inativacao' in df_novos.columns:
            df_novos['data_inativacao'] = pd.to_datetime(df_novos['data_inativacao'], errors='coerce')

        # Adicionar data_processamento atual
        df_novos['data_processamento'] = datetime.now()

        # Converter produtor_ativo para boolean se existir
        if 'produtor_ativo' in df_novos.columns:
            # Mapear valores para boolean
            df_novos['produtor_ativo'] = df_novos['produtor_ativo'].map({
                'Sim': True, 'sim': True, 'S': True, 's': True, True: True, 1: True, '1': True,
                'Não': False, 'não': False, 'N': False, 'n': False, False: False, 0: False, '0': False,
                None: None, np.nan: None
            })

        # Mostrar amostra dos dados
        print("\n📊 Amostra dos novos registros:")
        print(df_novos[['id_atendimento', 'nome_consultor', 'codigo_lr', 'data_solicitacao']].head())

        # Preparar para inserção
        df_prep = df_novos.copy()

        # Converter datas para formato ISO
        for col in ['data_solicitacao', 'data_inativacao', 'data_processamento']:
            if col in df_prep.columns and pd.api.types.is_datetime64_any_dtype(df_prep[col]):
                df_prep[col] = df_prep[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

        # Converter NaN para None
        df_prep = df_prep.replace({np.nan: None})
        
        # Remover colunas que não existem no Supabase
        colunas_apenas_local = ['id_composto']
        df_prep = df_prep.drop(columns=[c for c in colunas_apenas_local if c in df_prep.columns])

        # Converter para lista de dicionários
        registros = df_prep.to_dict(orient='records')

        # Inserir no Supabase
        print("\n🔄 Inserindo registros no Supabase...")

        # Definir tamanho do lote
        lote = 100
        total = len(registros)
        total_lotes = (total + lote - 1) // lote

        # Contadores
        sucesso = 0
        erro = 0

        inicio = datetime.now()

        # Processar em lotes
        for i in range(0, total, lote):
            lote_atual = registros[i:i+lote]
            num_lote = i // lote + 1

            try:
                print(f"Processando lote {num_lote}/{total_lotes} ({len(lote_atual)} registros)...")

                # Realizar upsert
                resultado = supabase.table(TAB_INATIVACAO_PRODUTORES).upsert(lote_atual,on_conflict="id_atendimento").execute()
                
                # Verificar resultado
                if hasattr(resultado, 'error') and resultado.error:
                    print(f"❌ Erro no lote {num_lote}: {resultado.error}")
                    erro += len(lote_atual)
                else:
                    sucesso += len(lote_atual)
                    print(f"✅ Lote {num_lote} processado com sucesso.")

                # Pausa para não sobrecarregar a API
                if num_lote < total_lotes:
                    time.sleep(0.5)

            except Exception as e:
                print(f"❌ Erro no lote {num_lote}: {str(e)}")
                erro += len(lote_atual)

        # Mostrar resumo
        fim = datetime.now()
        duracao = (fim - inicio).total_seconds()
        duracao_total = (fim - inicio_total).total_seconds()

        print("\n=== RESUMO DO ETL ===")
        print(f"📄 Arquivo processado: {arquivo_mais_recente}")
        print(f"📊 Total de registros no arquivo: {len(df_inativacao)}")
        print(f"🔍 Registros novos identificados: {len(df_novos)}")
        print(f"✅ Registros inseridos com sucesso: {sucesso}")
        print(f"❌ Registros com erro: {erro}")
        print(f"⏱️ Tempo de inserção: {duracao:.2f} segundos")
        print(f"⏱️ Tempo total do ETL: {duracao_total:.2f} segundos")
        print(f"📅 Data/hora de conclusão: {fim.strftime('%Y-%m-%d %H:%M:%S')}")

        return True

    except Exception as e:
        print(f"❌ Erro durante a execução: {str(e)}")
        traceback.print_exc()
        return False

## Execução: Inativação de Produtores


In [ ]:
# Executar ETL de Inativação de Produtores
resultado_inativacao = etl_inativacao(DIR_BD_SQ)

if resultado_inativacao:
    print('✅ ETL de inativação de produtores concluído com sucesso!')
else:
    print('❌ ETL de inativação de produtores falhou ou não identificou novos registros.')

# 3. Inativação de Consultores


In [ ]:
def etl_inativacao_consultor(diretorio=None, DIRETORIO_ENV=None):
    """
    ETL para identificar e registrar inativações de consultores no SmartQuestion.
    1. Lê o arquivo BD_STATUS_USUARIO_SQ.xlsx
    2. Identifica transições Sim → Não por consultor
    3. Faz upsert na tab_inativacao_consultor_sq_backup
    """
    try:
        print("=== ETL INATIVAÇÃO DE CONSULTOR ===")
        inicio_total = datetime.now()

        # ETAPA 1: IMPORTAR ARQUIVO
        print("\n🔍 ETAPA 1: IMPORTANDO ARQUIVO")

        diretorio_alvo = Path(diretorio) if diretorio else DIR_BD_SQ
        print(f"Diretório dos arquivos: {diretorio_alvo}")
        padrao = str(diretorio_alvo / "*BD_STATUS_USUARIO_SQ.xlsx")
        arquivos = glob.glob(padrao)

        if not arquivos:
            print(f"❌ Nenhum arquivo encontrado com o padrão {padrao}")
            return False

        arquivo = sorted(arquivos, key=os.path.getmtime, reverse=True)[0]
        print(f"✅ Arquivo encontrado: {arquivo}")

        df = pd.read_excel(arquivo)
        df["historyCreationDate"] = pd.to_datetime(df["historyCreationDate"])


        # ETAPA 2: IDENTIFICAR INATIVAÇÕES (Sim → Não)
        print("\n🔍 ETAPA 2: IDENTIFICANDO INATIVAÇÕES")

        df = df.sort_values(by=["Nome", "historyCreationDate"]).reset_index(drop=True)
        df["Ativo_anterior"] = df.groupby("Nome")["Ativo"].shift(1)

        df_inativacoes = df[
            (df["historyType"] == "UPDATED") &
            (df["Ativo"] == "Não") &
            (df["Ativo_anterior"] == "Sim")
        ][["historyCreationDate", "Nome"]].copy()

        df_inativacoes.rename(columns={
            "historyCreationDate": "data_inativacao",
            "Nome": "nome_consultor"
        }, inplace=True)

        print(f"✅ {len(df_inativacoes)} inativações identificadas")

        if len(df_inativacoes) == 0:
            print("⚠️ Nenhuma inativação encontrada no arquivo.")
            return True

        # ETAPA 3: GERAR HASH E PREPARAR DADOS
        print("\n🔄 ETAPA 3: PREPARANDO DADOS")

        def gerar_hash(nome, data):
            chave = f"{nome}_{data.date()}"
            return hashlib.md5(chave.encode()).hexdigest()

        df_inativacoes["id_inativacao_consultor"] = df_inativacoes.apply(
            lambda row: gerar_hash(row["nome_consultor"], row["data_inativacao"]),
            axis=1
        )

        df_inativacoes["data_inativacao"]   = df_inativacoes["data_inativacao"].dt.strftime('%Y-%m-%d')
        df_inativacoes["data_modificacao"]  = datetime.now(timezone.utc).isoformat()

        print(df_inativacoes[["nome_consultor", "data_inativacao", "id_inativacao_consultor"]].head())

        # ETAPA 4: UPSERT NO SUPABASE
        print("\n🔄 ETAPA 4: INSERINDO NO SUPABASE")

        url = os.getenv('SUPABASE_URL')
        key = os.getenv('SUPABASE_SERVICE_KEY')

        if not url or not key:
            print("❌ Credenciais não encontradas!")
            return False

        supabase = create_client(url, key)

        registros = df_inativacoes[[
            "id_inativacao_consultor",
            "nome_consultor",
            "data_inativacao",
            "data_modificacao"
        ]].to_dict(orient="records")

        lote       = 100
        total      = len(registros)
        total_lotes = (total + lote - 1) // lote
        sucesso = erro = 0

        for i in range(0, total, lote):
            lote_atual = registros[i:i+lote]
            num_lote   = i // lote + 1
            try:
                resultado = supabase.table(TAB_INATIVACAO_CONSULTORES).upsert(
                    lote_atual,
                    on_conflict="id_inativacao_consultor"
                ).execute()
                sucesso += len(lote_atual)
                print(f"✅ Lote {num_lote}/{total_lotes} inserido.")
            except Exception as e:
                print(f"❌ Erro no lote {num_lote}: {str(e)}")
                erro += len(lote_atual)

            if num_lote < total_lotes:
                time.sleep(0.5)

        # RESUMO
        fim = datetime.now()
        print("\n=== RESUMO ===")
        print(f"📄 Arquivo: {arquivo}")
        print(f"✅ Inseridos com sucesso: {sucesso}")
        print(f"❌ Com erro: {erro}")
        print(f"⏱️ Tempo total: {(fim - inicio_total).total_seconds():.2f}s")

        return True

    except Exception as e:
        print(f"❌ Erro: {str(e)}")
        traceback.print_exc()
        return False

## Execução: Inativação de Consultores


In [ ]:
# Executar ETL de Inativação de Consultores
resultado_inativacao_consultor = etl_inativacao_consultor(DIR_BD_SQ)

if resultado_inativacao_consultor:
    print('✅ ETL de inativação de consultores concluído com sucesso!')
else:
    print('❌ ETL de inativação de consultores falhou!')

# 4. Vínculos de Atendimento


In [ ]:
# Função melhorada para criar ID composto
def criar_id_composto(row):
    """
    Cria um ID composto usando MD5 hash de campos-chave com tratamento consistente
    """
    # Função auxiliar para tratar valores antes de concatenar
    def tratar_valor(valor):
        if pd.isna(valor) or valor is None:
            return ''
        # Converter para string, remover espaços extras e converter para minúsculas
        return str(valor).lower().strip()

    # Concatenar campos principais com tratamento consistente
    campos = [
        tratar_valor(row['codigo_lr']),
        # Tratar datas de forma especial para garantir formato consistente
        tratar_valor(pd.to_datetime(row['data_associacao']).strftime('%Y-%m-%d') if pd.notna(row['data_associacao']) else ''),
        tratar_valor(row['grupo_atendimento']),
        tratar_valor(row['projeto'])
    ]

    # Juntar campos e criar hash MD5
    texto_composto = '|'.join(campos)
    return hashlib.md5(texto_composto.encode('utf-8')).hexdigest()

# ✅ FUNÇÃO REUTILIZÁVEL DE TRIM
def aplicar_trim_colunas_texto(df, nome_df="DataFrame"):
    """
    Detecta automaticamente todas as colunas texto (object)
    e aplica TRIM, removendo espaços no início e fim.
    Retorna o DataFrame corrigido e um relatório das colunas tratadas.
    """
    # Identificar colunas texto automaticamente
    colunas_texto = df.select_dtypes(include=['object']).columns.tolist()

    print(f"🔄 Aplicando TRIM em {len(colunas_texto)} colunas texto de [{nome_df}]...")

    registros_corrigidos = 0

    for col in colunas_texto:
        # Contar quantos registros têm espaço antes/depois
        tem_espaco = df[col].apply(
            lambda x: isinstance(x, str) and x != x.strip()
        ).sum()

        if tem_espaco > 0:
            df[col] = df[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )
            print(f"  ✅ {col}: {tem_espaco} registro(s) corrigido(s)")
            registros_corrigidos += tem_espaco

    if registros_corrigidos == 0:
        print("  ✅ Nenhum espaço encontrado. Dados já estão limpos.")
    else:
        print(f"  📊 Total de correções: {registros_corrigidos}")

    return df
    
def etl_vinculos(diretorio=None, arquivo="BD_BI_VINCULOS_COMPLETO.xlsx", apenas_testar=False):
    """
    Função ETL para atualizar a tabela de vínculos no Supabase
    """
    try:
        print("=== ETL DE VÍNCULOS ===")
        inicio_total = datetime.now()

        # ETAPA 1: IMPORTAR ARQUIVOS EXCEL
        print("\n🔍 ETAPA 1: IMPORTANDO ARQUIVOS EXCEL")

        # Obter o diretório atual onde o notebook está sendo executado
        NOTEBOOK_DIR = DIR_BD_SQ

        # Definir o diretório
        diretorio_alvo = Path(diretorio) if diretorio else DIR_BD_SQ
        # os.chdir gerido no setup
        print(f"Diretório dos arquivos: {diretorio_alvo}")

        vinculo_arquivo_atual = diretorio_alvo / arquivo

        try:
            # Importar os arquivos Excel
            print("Importando arquivo de vínculos atuais...")
            df_vinculos = pd.read_excel(vinculo_arquivo_atual, engine='openpyxl')
            print(f"✅ Arquivo atual importado: {len(df_vinculos)} registros")


            # Mapear nomes de colunas
            vinculos_mapping = {
                'Código LR': 'codigo_lr',
                'Código agroindústria': 'codigo_agroindustria',
                'Código da fazenda':'codigo_fazenda',
                'Produtor(a)': 'nome_produtor',
                'Nome da propriedade': 'nome_propriedade',
                'Unidade de atendimento': 'unidade_atendimento',
                'Tipo de ponto atendimento': 'tipo_ponto_atendimento',
                'Cidade': 'cidade_produtor',
                'Estado': 'estado_produtor',
                'Ativo': 'vinculo_ativo',
                'Data de associação': 'data_associacao',
                'Grupo de atendimento': 'grupo_atendimento',
                'Consultor(a) no grupo de atendimento': 'consultor_grupo_atendimento',
                'PROJETO': 'projeto'
            }

            # Renomear colunas
            df_vinculos.rename(columns=vinculos_mapping, inplace=True)
            
            # Função para extrair projeto da coluna consultor_grupo_atendimento usando lógica de fórmula Excel
            def extrair_projeto_da_formula_excel(consultor_texto):
                if not isinstance(consultor_texto, str) or pd.isna(consultor_texto):
                    return None
            
                consultor_texto = consultor_texto.strip()
                try:
                    inicio = consultor_texto.find('(')
                    if inicio == -1:
                        return None
                    inicio += 1
            
                    fim = consultor_texto.find(')', inicio)
                    if fim == -1:
                        return None
            
                    projeto_extraido = consultor_texto[inicio:fim].strip()
                    if projeto_extraido == "":
                        return None
                    return projeto_extraido.upper()
                except Exception:
                    return None
                    
            # Extrair da coluna de grupo_atendimento o projeto
            df_vinculos['projeto'] =df_vinculos['grupo_atendimento'].apply(extrair_projeto_da_formula_excel)

            # Verificar se todas as colunas necessárias existem
            colunas_necessarias = ['codigo_lr', 'nome_produtor', 'nome_propriedade', 
                                 'consultor_grupo_atendimento', 'data_associacao', 
                                 'grupo_atendimento', 'projeto']

            colunas_faltantes = [col for col in colunas_necessarias if col not in df_vinculos.columns]
            if colunas_faltantes:
                print(f"⚠️ Colunas faltantes: {', '.join(colunas_faltantes)}")
                # Criar colunas faltantes com valores nulos
                for col in colunas_faltantes:
                    df_vinculos[col] = None

            print(f"✅ Arquivo importado com sucesso: {len(df_vinculos)} registros")

        except Exception as e:
            print(f"❌ Erro ao importar arquivos: {str(e)}")
            # os.chdir gerido no setup
            return False

        # Voltar ao diretório de ambiente
        # os.chdir gerido no setup

        if len(df_vinculos) == 0:
            print("❌ Nenhum dado encontrado nos arquivos.")
            return False

        # ETAPA 2: VERIFICAÇÃO E TRATAMENTO DE DUPLICATAS
        print("\n🔍 ETAPA 2: VERIFICAÇÃO E TRATAMENTO DE DUPLICATAS")

        # Verificar valores nulos em colunas críticas
        colunas_criticas = ['codigo_lr', 'nome_produtor', 'nome_propriedade', 'consultor_grupo_atendimento', 'projeto']
        print("Verificando valores nulos em colunas críticas:")
        for col in colunas_criticas:
            nulos = df_vinculos[col].isna().sum()
            vazios = (df_vinculos[col] == '').sum() if df_vinculos[col].dtype == 'object' else 0
            print(f"  - Coluna {col}: {nulos} valores nulos, {vazios} strings vazias")


        # Se o id_composto deve ser sensível APENAS à data, use .dt.normalize()
        df_vinculos['data_associacao'] = pd.to_datetime(df_vinculos['data_associacao'], errors='coerce')
        # Preencher NaT com um valor padrão ou None para o hash
        df_vinculos['data_associacao'] = df_vinculos['data_associacao'].fillna(pd.NaT) # Usar NaT para representar nulo de data
        
        # Verificar duplicatas no DataFrame concatenado
        print("\nVerificando duplicatas no DataFrame...")
        duplicatas = df_vinculos[df_vinculos.duplicated(subset=['codigo_lr', 'consultor_grupo_atendimento', 'data_associacao'], keep=False)] # Pode ser que houve 2 vínculos para o produtor-consultor em datas diferentes.
        print(f"Encontradas {len(duplicatas)} linhas duplicadas (considerando código LR e consultor)")

        if not duplicatas.empty:
            # Mostrar algumas duplicatas para análise
            print("\nExemplo de duplicatas:")
            for codigo_lr in duplicatas['codigo_lr'].unique()[:3]:  # Mostrar até 3 exemplos
                registros_duplicados = df_vinculos[df_vinculos['codigo_lr'] == codigo_lr]
                print(f"\nCódigo LR: {codigo_lr}")
                print(registros_duplicados[['codigo_lr', 'nome_produtor', 'consultor_grupo_atendimento', 'projeto']].head())

            # Resolver duplicatas priorizando valores não nulos
            print("\n🔄 Resolvendo duplicatas...")

            # Função para mesclar valores, priorizando não nulos
            def mesclar_valores(serie):
                # Remover valores nulos
                valores_validos = [v for v in serie if pd.notna(v) and v != '']
                # Se todos são nulos, retornar None
                if not valores_validos:
                    return None
                # Caso contrário, retornar o primeiro valor não nulo
                return valores_validos[0]

            # Agrupar por código LR e consultor, mesclando valores
            colunas_agrupar = ['codigo_lr', 'consultor_grupo_atendimento', 'data_associacao']
            colunas_mesclar = df_vinculos.columns.difference(colunas_agrupar)

            df_vinculos = df_vinculos.groupby(colunas_agrupar)[colunas_mesclar].agg(mesclar_valores).reset_index()

            print(f"DataFrame após resolução de duplicatas: {len(df_vinculos)} registros")

            # Verificar se as duplicatas foram resolvidas
            duplicatas_restantes = df_vinculos[df_vinculos.duplicated(subset=['codigo_lr', 'consultor_grupo_atendimento', 'data_associacao'], keep=False)]
            print(f"Duplicatas restantes: {len(duplicatas_restantes)}")

        # ETAPA 3: TRATAMENTO DA COLUNA PROJETO
        print("\n🔍 ETAPA 3: TRATAMENTO DA COLUNA PROJETO")

        # Verificar valores nulos na coluna projeto ANTES do tratamento
        nulos_projeto_antes = df_vinculos['projeto'].isna().sum()
        vazios_projeto_antes = (df_vinculos['projeto'] == '').sum()
        print(f"Valores nulos na coluna projeto ANTES do tratamento: {nulos_projeto_antes}")
        print(f"Strings vazias na coluna projeto ANTES do tratamento: {vazios_projeto_antes}")

        # Verificar um caso específico para confirmar o tratamento
        codigo_lr_problema = 'LR03243'
        registro_problema = df_vinculos[df_vinculos['codigo_lr'] == codigo_lr_problema]
        if not registro_problema.empty:
            print(f"\nVerificação do registro problemático {codigo_lr_problema}:")
            print(registro_problema[['codigo_lr', 'nome_produtor', 'consultor_grupo_atendimento', 'projeto']].head())

        # ETAPA 4: PREPARAÇÃO DOS DADOS E CRIAÇÃO DO ID COMPOSTO
        print("\n🔄 ETAPA 4: PREPARAÇÃO DOS DADOS E CRIAÇÃO DO ID COMPOSTO")

        # Ajustar tipos de dados
        print("🔄 Ajustando tipos de dados...")

        # Converter data_associacao para datetime
        if 'data_associacao' in df_vinculos.columns:
            df_vinculos['data_associacao'] = pd.to_datetime(df_vinculos['data_associacao'], errors='coerce')

        # Converter vinculo_ativo para boolean
        if 'vinculo_ativo' in df_vinculos.columns:
            # Mapear valores para boolean
            df_vinculos['vinculo_ativo'] = df_vinculos['vinculo_ativo'].map({
                'Sim': True, 'sim': True, 'S': True, 's': True, 'Ativo': True, 'ativo': True, 
                'True': True, 'true': True, 'Verdadeiro': True, 'verdadeiro': True, 
                'V': True, 'v': True, '1': True, 1: True, True: True,

                'Não': False, 'não': False, 'N': False, 'n': False, 'Inativo': False, 'inativo': False, 
                'False': False, 'false': False, 'Falso': False, 'falso': False, 
                'F': False, 'f': False, '0': False, 0: False, False: False,

                None: None, np.nan: None
            })

        # ✅ TRIM AUTOMÁTICO: substitui todo o bloco manual anterior
        df_vinculos = aplicar_trim_colunas_texto(df_vinculos, nome_df="df_vinculos")
            
        # Criar coluna de ID composto
        print("🔄 Criando ID composto para cada registro...")
        df_vinculos['id_composto'] = df_vinculos.apply(criar_id_composto, axis=1)

        # Verificar se há duplicatas no ID composto
        duplicatas_id = df_vinculos[df_vinculos.duplicated(subset=['id_composto'], keep=False)]
        if not duplicatas_id.empty:
            print(f"⚠️ Encontradas {len(duplicatas_id)} duplicatas de ID composto!")
            # Manter apenas a primeira ocorrência de cada ID composto
            df_vinculos = df_vinculos.drop_duplicates(subset=['id_composto'], keep='first')
            print(f"✅ Duplicatas removidas. DataFrame final: {len(df_vinculos)} registros")

        # Adicionar data_processamento atual
        df_vinculos['data_processamento'] = datetime.now()

        # ETAPA 5: BUSCAR REGISTROS EXISTENTES NO SUPABASE
        print("\n🔍 ETAPA 5: BUSCANDO REGISTROS EXISTENTES NO SUPABASE")

        # Carregar credenciais do Supabase
        key = os.getenv('SUPABASE_SERVICE_KEY')
        url = os.getenv('SUPABASE_URL')

        if not key or not url:
            print("❌ Credenciais não encontradas!")
            return False

        # Inicializar cliente Supabase
        supabase = create_client(url, key)

        # Verificar se a coluna id_composto existe na tabela
        print("🔍 Verificando se a coluna id_composto existe...")

        try:
            # Tentar buscar um registro com a coluna id_composto
            supabase.table(TAB_VINCULOS_STAGING).select("id_composto").limit(1).execute()
            coluna_existe = True
            print("✅ Coluna id_composto encontrada na tabela.")
        except Exception as e:
            coluna_existe = False
            print("⚠️ Coluna id_composto não existe na tabela. Será criada durante o processo.")

        # Buscar dados existentes para comparação
        print("🔍 Buscando registros existentes...")

        try:
            # Buscar os campos necessários para criar o ID composto
            # Na etapa 5, modificar a consulta para selecionar todas as colunas
            resultado = (
                supabase
                .table(TAB_VINCULOS_STAGING) 
                .select("*")   # Selecionar todas as colunas
                .execute()
            )

            if resultado.data:
                # Criar DataFrame com os registros existentes
                df_existentes = pd.DataFrame(resultado.data)

                # Criar ID composto para os registros existentes usando a mesma função
                df_existentes['id_composto'] = df_existentes.apply(criar_id_composto, axis=1)

                # Obter conjunto de IDs existentes
                ids_existentes = set(df_existentes['id_composto'])

                print(f"✅ Encontrados {len(ids_existentes)} registros existentes no Supabase")

                # Verificar um caso específico para debug
                codigo_lr_problema = 'LR03243'
                registro_excel = df_vinculos[df_vinculos['codigo_lr'] == codigo_lr_problema]
                registro_supabase = df_existentes[df_existentes['codigo_lr'] == codigo_lr_problema]

                if not registro_excel.empty and not registro_supabase.empty:
                    print(f"\n🔍 VERIFICANDO REGISTRO ESPECÍFICO: {codigo_lr_problema}")
                    print("No Excel:")
                    print(registro_excel[['codigo_lr', 'nome_produtor', 'consultor_grupo_atendimento', 'projeto', 'id_composto', 'data_associacao']].iloc[0])
                    print("\nNo Supabase:")
                    print(registro_supabase[['codigo_lr', 'nome_produtor', 'consultor_grupo_atendimento', 'projeto', 'id_composto', 'data_associacao']].iloc[0])

                    # Verificar se os IDs compostos são iguais
                    id_excel = registro_excel['id_composto'].iloc[0]
                    id_supabase = registro_supabase['id_composto'].iloc[0]
                    if id_excel == id_supabase:
                        print(f"\n✅ IDs compostos correspondem: {id_excel}")
                    else:
                        print(f"\n❌ IDs compostos diferem:")
                        print(f"  Excel: {id_excel}")
                        print(f"  Supabase: {id_supabase}")

                        # Comparar os campos usados para gerar o ID
                        for campo in ['codigo_lr', 'nome_produtor', 'nome_propriedade', 'consultor_grupo_atendimento', 'grupo_atendimento', 'projeto']:
                            if campo in registro_excel and campo in registro_supabase:
                                valor_excel = registro_excel[campo].iloc[0]
                                valor_supabase = registro_supabase[campo].iloc[0]
                                if str(valor_excel).lower().strip() != str(valor_supabase).lower().strip():
                                    print(f"Diferença no campo '{campo}':")
                                    print(f"  Excel: '{valor_excel}' (tipo: {type(valor_excel)})")
                                    print(f"  Supabase: '{valor_supabase}' (tipo: {type(valor_supabase)})")
            else:
                ids_existentes = set()
                print("⚠️ Nenhum registro encontrado no Supabase.")
        except Exception as e:
            print(f"⚠️ Erro ao buscar registros existentes: {str(e)}")
            # Em caso de erro, assumir que não há registros
            ids_existentes = set()

        # ETAPA 6: IDENTIFICAR REGISTROS NOVOS OU ALTERADOS (CORRIGIDO PARA DATAS)
        print("\n🔍 ETAPA 6: IDENTIFICANDO REGISTROS NOVOS OU ALTERADOS")
        
        # Identificar registros novos ou alterados
        ids_atuais = set(df_vinculos['id_composto'])
        
        # Registros novos (não existem no Supabase)
        ids_novos = ids_atuais - ids_existentes
        df_novos = df_vinculos[df_vinculos['id_composto'].isin(ids_novos)]
        
        # Para os registros que existem em ambos, verificar se há diferenças
        ids_comuns = ids_atuais.intersection(ids_existentes)
        df_atualizar = pd.DataFrame()  # Iniciar com DataFrame vazio
        
        if ids_comuns and 'df_existentes' in locals() and not df_existentes.empty:
            print(f"Verificando diferenças em {len(ids_comuns)} registros comuns...")
        
            # Definir explicitamente as colunas a comparar (apenas colunas que existem em ambos os lugares)
            colunas_supabase = set(df_existentes.columns)
            colunas_excel = set(df_vinculos.columns)
            colunas_comuns = list(colunas_supabase.intersection(colunas_excel))
            colunas_comuns = [col for col in colunas_comuns if col not in ['id_composto', 'data_processamento']]
        
            print(f"Comparando apenas as {len(colunas_comuns)} colunas comuns: {', '.join(colunas_comuns)}")
        
            # Função melhorada para normalizar valores, com foco especial em datas
            def normalizar_valor(valor):
                """
                Normaliza valores para comparação consistente, com tratamento especial para datas
                """
                if pd.isna(valor) or valor is None:
                    return ''
                elif isinstance(valor, bool):
                    return 'true' if valor else 'false'
                elif isinstance(valor, (int, float)):
                    return str(float(valor))
                elif isinstance(valor, pd.Timestamp) or isinstance(valor, datetime):
                    # Para objetos de data/hora, retornar apenas a parte da data
                    return valor.strftime('%Y-%m-%d')
                elif isinstance(valor, str):
                    # Tratar strings que podem representar datas
                    valor_lower = valor.lower().strip()
        
                    # Verificar se a string parece uma data com hora (contém T ou t)
                    if 't' in valor_lower and len(valor_lower) > 10:
                        try:
                            # Extrair apenas a parte da data antes do T
                            data_parte = valor_lower.split('t')[0]
                            return data_parte
                        except:
                            return valor_lower
                    return valor_lower
                else:
                    return str(valor).lower().strip()
        
            # Converter df_existentes para dicionário para facilitar a busca
            dict_existentes = {}
            for _, row in df_existentes.iterrows():
                id_composto = row['id_composto']
                dict_existentes[id_composto] = row.to_dict()
        
            # Verificar diferenças para cada registro comum
            registros_diferentes = []
        
            for _, row in df_vinculos[df_vinculos['id_composto'].isin(ids_comuns)].iterrows():
                id_composto = row['id_composto']
        
                if id_composto in dict_existentes:
                    registro_existente = dict_existentes[id_composto]
        
                    # Verificar se há diferenças nas colunas de comparação
                    tem_diferenca = False
                    diferencas = []
        
                    for col in colunas_comuns:
                        # Tratamento especial para a coluna data_associacao
                        if col == 'data_associacao':
                            # Extrair apenas a data (YYYY-MM-DD) de ambos os valores
                            try:
                                # Para o valor atual
                                if pd.notna(row[col]):
                                    if isinstance(row[col], (pd.Timestamp, datetime)):
                                        valor_atual = row[col].strftime('%Y-%m-%d')
                                    else:
                                        valor_str = str(row[col]).lower()
                                        if 't' in valor_str:
                                            valor_atual = valor_str.split('t')[0]
                                        else:
                                            valor_atual = valor_str
                                else:
                                    valor_atual = ''
        
                                # Para o valor existente
                                if pd.notna(registro_existente[col]):
                                    if isinstance(registro_existente[col], (pd.Timestamp, datetime)):
                                        valor_existente = registro_existente[col].strftime('%Y-%m-%d')
                                    else:
                                        valor_str = str(registro_existente[col]).lower()
                                        if 't' in valor_str:
                                            valor_existente = valor_str.split('t')[0]
                                        else:
                                            valor_existente = valor_str
                                else:
                                    valor_existente = ''
                            except:
                                # Em caso de erro, usar a normalização padrão
                                valor_atual = normalizar_valor(row[col])
                                valor_existente = normalizar_valor(registro_existente[col])
                        else:
                            # Para outras colunas, usar a normalização padrão
                            valor_atual = normalizar_valor(row[col])
                            valor_existente = normalizar_valor(registro_existente[col])
        
                        # Comparar os valores normalizados
                        if valor_atual != valor_existente:
                            tem_diferenca = True
                            diferencas.append(f"{col}: '{valor_existente}' -> '{valor_atual}'")
        
                    if tem_diferenca:
                        # Adicionar informações de debug
                        print(f"Diferenças encontradas para ID {id_composto}:")
                        for diff in diferencas:
                            print(f"  - {diff}")
                        registros_diferentes.append(row)
        
            # Criar DataFrame com registros que realmente precisam ser atualizados
            if registros_diferentes:
                df_atualizar = pd.DataFrame(registros_diferentes)
                print(f"Encontrados {len(df_atualizar)} registros com diferenças reais que precisam ser atualizados.")
            else:
                print("✅ Nenhuma diferença encontrada nos registros comuns!")
        
        print(f"✅ Registros novos: {len(df_novos)}")
        print(f"✅ Registros a atualizar: {len(df_atualizar)}")
        print(f"✅ Total a processar: {len(df_novos) + len(df_atualizar)}")



        if apenas_testar:
            print("🧪 MODO TESTE (DRY-RUN) ATIVADO:")
            print("--------------------------------------------------")
            print(f"📊 Total de registros no Excel: {len(df_vinculos)}")
            print(f"🆕 Registros NOVOS identificados: {len(df_novos)}")
            print(f"🔄 Registros a ATUALIZAR identificados: {len(df_atualizar)}")
            if len(df_novos) > 0:
                print("🔍 Amostra dos 5 primeiros registros NOVOS:")
                cols_preview = [c for c in ['codigo_lr', 'nome_produtor', 'consultor_grupo_atendimento', 'projeto'] if c in df_novos.columns]
                print(df_novos[cols_preview].head())
            print("✅ Nenhuma alteração foi gravada no Supabase (Dry-run concluído com sucesso).")
            return True

        # ETAPA 7: INSERIR E ATUALIZAR NO SUPABASE
        print("\n🔄 ETAPA 7: PROCESSANDO REGISTROS NO SUPABASE")
        
        if len(df_novos) + len(df_atualizar) == 0:
            print("✅ Não há registros para processar.")
            return True
        
        # ── Função auxiliar de preparo
        def preparar_payload(df):
            df_prep = df.copy()
            for col in ['data_associacao', 'data_processamento']:
                if col in df_prep.columns and pd.api.types.is_datetime64_any_dtype(df_prep[col]):
                    df_prep[col] = df_prep[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
            df_prep = df_prep.replace({np.nan: None})
            return df_prep.to_dict(orient='records')
        
        LOTE = 100
        sucesso = 0
        erro = 0
        inicio = datetime.now()
        
        # ── BLOCO 1: INSERT — apenas registros NOVOS
        if not df_novos.empty:
            print(f"\n📥 Inserindo {len(df_novos)} registros NOVOS...")
            registros_novos = preparar_payload(df_novos)
        
            for i in range(0, len(registros_novos), LOTE):
                lote_atual = registros_novos[i:i+LOTE]
                num_lote = i // LOTE + 1
                total_lotes = (len(registros_novos) + LOTE - 1) // LOTE
                try:
                    supabase.table(TAB_VINCULOS_STAGING).insert(lote_atual).execute()
                    # ↑ INSERT puro: só para registros que não existem
                    sucesso += len(lote_atual)
                    print(f"  ✅ Lote {num_lote}/{total_lotes} inserido.")
                except Exception as e:
                    erro += len(lote_atual)
                    print(f"  ❌ Erro no lote {num_lote}: {e}")
                time.sleep(0.3)
        
        # ── BLOCO 2: UPDATE — apenas registros EXISTENTES com diferença
        if not df_atualizar.empty:
            print(f"\n🔄 Atualizando {len(df_atualizar)} registros EXISTENTES...")
            registros_atualizar = preparar_payload(df_atualizar)
        
            for registro in registros_atualizar:
                id_composto = registro.get('id_composto')
                try:
                    (
                        supabase
                        .table(TAB_VINCULOS_STAGING)
                        .update(registro)
                        .eq("id_composto", id_composto)
                        # ↑ UPDATE filtrado pelo id_composto: nunca gera conflito de constraint
                        .execute()
                    )
                    sucesso += 1
                except Exception as e:
                    erro += 1
                    print(f"  ❌ Erro ao atualizar {id_composto}: {e}")

        # ETAPA 8: RESUMO E FINALIZAÇÃO
        fim = datetime.now()
        duracao = (fim - inicio).total_seconds()
        duracao_total = (fim - inicio_total).total_seconds()

        print("\n=== RESUMO DO ETL DE VÍNCULOS ===")
        print(f"📄 Arquivos processados: {vinculo_arquivo_atual}")
        print(f"📊 Total de registros: {len(df_vinculos)}")
        print(f"🆕 Registros novos: {len(df_novos)}")
        print(f"🔄 Registros atualizados: {len(df_atualizar)}")
        print(f"✅ Registros processados com sucesso: {sucesso}")
        print(f"❌ Registros com erro: {erro}")
        print(f"⏱️ Tempo de inserção: {duracao:.2f} segundos")
        print(f"⏱️ Tempo total do ETL: {duracao_total:.2f} segundos")
        print(f"📅 Data/hora de conclusão: {fim.strftime('%Y-%m-%d %H:%M:%S')}")

        return True

    except Exception as e:
        print(f"❌ Erro durante a execução: {str(e)}")
        traceback.print_exc()
        return False

## Execução: Vínculos de Atendimento (Modo Teste)


In [ ]:
# Executar ETL de Vínculos em Modo Teste (apenas_testar=True)
resultado_vinculos = etl_vinculos(DIR_BD_SQ, apenas_testar=False)

if resultado_vinculos:
    print('✅ Teste de ETL de vínculos concluído com sucesso!')
else:
    print('❌ Teste de ETL de vínculos falhou!')

# 5. Visitas dos Projetos


In [ ]:
def processar_relatorios_visitas(diretorio_dados=None):
    """
    Processa os relatórios individuais de visitas e cria um DataFrame consolidado

    Args:
        diretorio_dados: Diretório onde estão os arquivos de relatórios

    Returns:
        DataFrame consolidado com os dados de visitas
    """
    try:
        print("=== PROCESSAMENTO DE RELATÓRIOS DE VISITAS ===")
        inicio = datetime.now()

        # Mudar para o diretório especificado se fornecido
        if diretorio_dados:
            # os.chdir gerido no setup
            print(f"Diretório alterado para: {diretorio_dados}")
        
        # Definir nomes dos arquivos
        regenera_arquivo = config_yaml['smartquestion']['arquivos']['visita_regenera']
        alvoar_arquivo = config_yaml['smartquestion']['arquivos']['visita_alvoar']
        semear_arquivo = config_yaml['smartquestion']['arquivos']['visita_semear']
        ccpr_arquivo = config_yaml['smartquestion']['arquivos']['visita_ccpr']
        lpa_arquivo = config_yaml['smartquestion']['arquivos']['visita_lpa']
        
        # Definir configurações de importação
        # Alvoar
        alvoar_columns = [0,1,2,3,4,5,8,9,10,11,12,13,14,17,18,19,20,21,22,23,24,31,32]
        alvoar_aba = 'INF_GERAIS'
        alvoar_inicio = 2

        # Regenera
        regenera_columns = [1,2,3,4,5,6,9,11,12,13,14,15,16,17,18,19,20,21,28,30,32,33]
        regenera_aba = 'DADOS_COLETADOS'
        regenera_inicio = 3
        
        # Semear
        semear_columns = [1,2,3,4,5,7,8,9,10,11,12,13,14,15,16,17,18,25,27,29,30]
        semear_aba = 'DADOS_COLETADOS'
        semear_inicio = 3
        
        # CCPR
        ccpr_columns = [1,2,3,4,5,7,8,9,10,11,12,13,14,15,22,24,26,27]
        ccpr_aba = 'DADOS_COLETADOS'
        ccpr_inicio = 3
        
        # LPA
        lpa_columns = [0,1,2,3,4,6,8,9,10,11,12,13,14,17,18,19,20,21,22,23,24]
        lpa_aba = 'INF_GERAIS'
        lpa_inicio = 2
        
        # Dicionário de mapeamento de colunas
        col_mapping = {
            'alvoar':{
                'Produtor(a):':'nome_produtor',
                'Código do produtor(a)':'codigo_lr',
                'Propriedade:':'nome_propriedade',
                'Consultor(a):':'nome_consultor',
                'Número de atendimento:':'id_atendimento',
                'Data fim no sistema:':'data_visita',
                'Área utilizada para pecuária (hectares): ':'area_pecuaria_ha',
                'Quantidade de dias-homem (média diária): ':'mdo_dias_homem',
                'Produção (litros/dia): ':'producao_l_dia',
                'CCS mensal (x 1.000 cél./ml): ':'ccs_mensal',
                'CPP mensal (x 1.000 UFC/ml): ':'cpp_mensal',
                'Gordura mensal (%): ':'gordura_mensal',
                'Proteína mensal (%): ':'proteina_mensal',
                'Vacas em lactação (cabeças): ':'vacas_lactacao',
                'Vacas secas (cabeças): ':'vacas_secas',
                'Bezerras em aleitamento (cabeças): ':'bezerras_aleitamento',
                'Bezerros em aleitamento (cabeças): ':'bezerros_aleitamento',
                'Novilhas (cabeças): ':'novilhas',
                'Reprodutores (cabeças): ':'reprodutores',
                'Receptoras (cabeças): ':'receptoras',
                'Total de rebanho (cabeças): ':'rebanho_total',
                'Valor pago pelo produtor (R$):':'valor_pago_produtor',
                'Valor subsidiado pela Alvoar (R$):':'valor_pago_agroindustria'
            },
            'semear':{
                'Número do atendimento':'id_atendimento',
                'Consultor(a)':'nome_consultor',
                'Código do produtor(a)':'codigo_lr',
                'Nome do produtor(a)':'nome_produtor',
                'Propriedade':'nome_propriedade',
                'Data da realização da visita:':'data_visita',
                'Área utilizada para pecuária (hectares):':'area_pecuaria_ha',
                'Quantidade de dias-homem (média diária):':'mdo_dias_homem',
                'Produção (litros/dia):':'producao_l_dia',
                'Vacas em lactação (cabeças):':'vacas_lactacao',
                'Vacas secas (cabeças):':'vacas_secas',
                'Bezerras em aleitamento (cabeças):':'bezerras_aleitamento',
                'Bezerros em aleitamento (cabeças):':'bezerros_aleitamento',
                'Novilhas (cabeças):':'novilhas',
                'Reprodutores (cabeças):':'reprodutores',
                'Receptoras (cabeças):':'receptoras',
                'Total de rebanho (cabeças):':'rebanho_total',
                'CCS mensal (x1.000 cél./ml):':'ccs_mensal',
                'CPP mensal (x1.000 UFC/ml):':'cpp_mensal',
                'Gordura mensal (%):':'gordura_mensal',
                'Proteína mensal (%):':'proteina_mensal'
            },
            'regenera':{
                'Número do atendimento':'id_atendimento',
                'Consultor(a)':'nome_consultor',
                'Código do produtor(a)':'codigo_lr',
                'ID Farm':'id_farm',
                'Nome do produtor(a)':'nome_produtor',
                'Propriedade':'nome_propriedade',
                'Data da realização da visita:':'data_visita',
                'Área utilizada para pecuária (hectares):':'area_pecuaria_ha',
                'Quantidade de dias-homem (média diária):':'mdo_dias_homem',
                'Produção (litros/dia):':'producao_l_dia',
                'Vacas em lactação (cabeças):':'vacas_lactacao',
                'Vacas secas (cabeças):':'vacas_secas',
                'Bezerras em aleitamento (cabeças):':'bezerras_aleitamento',
                'Bezerros em aleitamento (cabeças):':'bezerros_aleitamento',
                'Novilhas (cabeças):':'novilhas',
                'Reprodutores (cabeças):':'reprodutores',
                'Receptoras (cabeças):':'receptoras',
                'Total de rebanho (cabeças):':'rebanho_total',
                'CCS mensal (x1.000 cél./ml):':'ccs_mensal',
                'CPP mensal (x1.000 UFC/ml):':'cpp_mensal',
                'Gordura mensal (%):':'gordura_mensal',
                'Proteína mensal (%):':'proteina_mensal'
            },
            'ccpr':{
                'Número do atendimento':'id_atendimento',
                'Consultor(a)':'nome_consultor',
                'Código do produtor(a)':'codigo_lr',
                'Nome do produtor(a)':'nome_produtor',
                'Propriedade':'nome_propriedade',
                'Data da realização da visita:.1':'data_visita',
                'Área utilizada para pecuária (hectares):':'area_pecuaria_ha',
                'Quantidade de dias-homem (média diária):':'mdo_dias_homem',
                'Produção (litros/dia):':'producao_l_dia',
                'Vacas em lactação (cabeças):':'vacas_lactacao',
                'Vacas secas (cabeças):':'vacas_secas',
                'Bezerros em aleitamento (cabeças):':'bezerros_aleitamento',
                'Novilhas (cabeças):':'novilhas',
                'Reprodutores (cabeças):':'reprodutores',
                'Receptoras (cabeças):':'receptoras',
                'Total de rebanho (cabeças):':'rebanho_total',
                'CCS mensal (x1.000 cél./ml):':'ccs_mensal',
                'CPP mensal (x1.000 UFC/ml):':'cpp_mensal',
                'Gordura mensal (%):':'gordura_mensal',
                'Proteína mensal (%):':'proteina_mensal'
            },
            'lpa':{
                'Produtor(a):':'nome_produtor',
                'Código do produtor(a)':'codigo_lr',
                'Propriedade:':'nome_propriedade',
                'Consultor(a):':'nome_consultor',
                'Número de atendimento:':'id_atendimento',
                'Data da realização da visita:':'data_visita',
                'Área utilizada para pecuária (hectares): ':'area_pecuaria_ha',
                'Quantidade de dias-homem (média diária): ':'mdo_dias_homem',
                'Produção (litros/dia): ':'producao_l_dia',
                'CCS mensal (x 1.000 cél./ml): ':'ccs_mensal',
                'CPP mensal (x 1.000 UFC/ml): ':'cpp_mensal',
                'Gordura mensal (%): ':'gordura_mensal',
                'Proteína mensal (%): ':'proteina_mensal',
                'Vacas em lactação (cabeças): ':'vacas_lactacao',
                'Vacas secas (cabeças): ':'vacas_secas',
                'Bezerras em aleitamento (cabeças): ':'bezerras_aleitamento',
                'Bezerros em aleitamento (cabeças): ':'bezerros_aleitamento',
                'Novilhas (cabeças): ':'novilhas',
                'Reprodutores (cabeças): ':'reprodutores',
                'Receptoras (cabeças): ':'receptoras',
                'Total de rebanho (cabeças): ':'rebanho_total'
            }
        }
        
        # Lista para armazenar os DataFrames
        list_df = []
        
        # Importar cada um dos relatórios de visitas
        print("\n🔍 ETAPA 1: IMPORTANDO RELATÓRIOS DE VISITAS")
        
        # Regenera
        try:
            df_regenera = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / regenera_arquivo, sheet_name=regenera_aba, header=regenera_inicio, usecols=regenera_columns)
            df_regenera.rename(columns=col_mapping['regenera'], inplace=True)
            df_regenera['origem_dados'] = 'REGENERA'
            df_regenera['valor_pago_produtor'] = np.nan
            df_regenera['valor_pago_agroindustria'] = np.nan
            list_df.append(df_regenera)
            print(f"✅ Relatório de visitas do Regenera lido com sucesso!! ({len(df_regenera)} registros)")
        except Exception as e:
            print(f"⚠️ Erro ao importar relatório Regenera: {str(e)}")
        
        # Alvoar
        try:
            df_alvoar = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / alvoar_arquivo, sheet_name=alvoar_aba, header=alvoar_inicio, usecols=alvoar_columns)
            df_alvoar.rename(columns=col_mapping['alvoar'], inplace=True)
            df_alvoar['origem_dados'] = 'ALVOAR'
            df_alvoar['data_visita'] = df_alvoar['data_visita'].dt.normalize()
            df_alvoar['id_farm'] = np.nan
            list_df.append(df_alvoar)
            print(f"✅ Relatório de visitas do Alvoar lido com sucesso!! ({len(df_alvoar)} registros)")
        except Exception as e:
            print(f"⚠️ Erro ao importar relatório Alvoar: {str(e)}")
        
        # Semear
        try:
            df_semear = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / semear_arquivo, sheet_name=semear_aba, header=semear_inicio, usecols=semear_columns)
            df_semear.rename(columns=col_mapping['semear'], inplace=True)
            df_semear['origem_dados'] = 'SEMEAR'
            df_semear['id_farm'] = np.nan
            df_semear['valor_pago_produtor'] = np.nan
            df_semear['valor_pago_agroindustria'] = np.nan            
            list_df.append(df_semear)
            print(f"✅ Relatório de visitas do Semear lido com sucesso!! ({len(df_semear)} registros)")
        except Exception as e:
            print(f"⚠️ Erro ao importar relatório Semear: {str(e)}")
        
        
        # CCPR
        try:
            df_ccpr = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / ccpr_arquivo, sheet_name=ccpr_aba, header=ccpr_inicio, usecols=ccpr_columns)
            df_visita_ccpr = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / ccpr_arquivo, sheet_name='DADOS_DA_VISITA', header=4, usecols=['Número do atendimento', 'Data da realização da visita:.1'])
            df_ccpr = pd.merge(df_ccpr, df_visita_ccpr, how='left', on=['Número do atendimento'])
            df_ccpr.rename(columns=col_mapping['ccpr'], inplace=True)
            df_ccpr['bezerras_aleitamento'] = np.nan
            df_ccpr['origem_dados'] = 'CCPR'
            df_ccpr['id_farm'] = np.nan
            df_ccpr['valor_pago_produtor'] = np.nan
            df_ccpr['valor_pago_agroindustria'] = np.nan 
            list_df.append(df_ccpr)
            print(f"✅ Relatório de visitas do CCPR lido com sucesso!! ({len(df_ccpr)} registros)")
        except Exception as e:
            print(f"⚠️ Erro ao importar relatório CCPR: {str(e)}")
        
        # LPA
        try:
            df_lpa = pd.read_excel(Path(diretorio_dados or DIR_BD_SQ) / lpa_arquivo, sheet_name=lpa_aba, header=lpa_inicio, usecols=lpa_columns)
            df_lpa.rename(columns=col_mapping['lpa'], inplace=True)
            df_lpa['origem_dados'] = 'LPA'
            df_lpa['id_farm'] = np.nan
            df_lpa['valor_pago_produtor'] = np.nan
            df_lpa['valor_pago_agroindustria'] = np.nan
            list_df.append(df_lpa)
            print(f"✅ Relatório de visitas do LPA lido com sucesso!! ({len(df_lpa)} registros)")
        except Exception as e:
            print(f"⚠️ Erro ao importar relatório LPA: {str(e)}")

        # Verificar se há DataFrames para processar
        if not list_df:
            print("❌ Nenhum relatório de visitas foi importado com sucesso.")
            # os.chdir gerido no setup
            return None

        print(f"✅ Total de relatórios importados: {len(list_df)}")

        # ETAPA 2: CONSOLIDAR E TRATAR OS DADOS
        print("\n🔄 ETAPA 2: CONSOLIDANDO E TRATANDO OS DADOS")

        # Obter todas as colunas únicas de todos os dataframes
        all_columns = set()
        for df in list_df:
            all_columns.update(df.columns)

        print(f"Total de colunas únicas: {len(all_columns)}")

        # Adicionar colunas faltantes em cada dataframe
        for df in list_df:
            for col in all_columns:
                if col not in df.columns:
                    df[col] = None

        # Concatenar todos os dataframes
        df_visitas = pd.concat(list_df, ignore_index=True)

        print(f"DataFrame consolidado criado com sucesso!")
        print(f"Total de registros: {len(df_visitas)}")
        print(f"Total de colunas: {len(df_visitas.columns)}")

        # Converter a coluna de data para o tipo datetime
        if 'data_visita' in df_visitas.columns:
            df_visitas['data_visita'] = pd.to_datetime(df_visitas['data_visita'], errors='coerce')

        # Converter colunas numéricas para o tipo float
        numeric_columns = [
            'area_pecuaria_ha', 'mdo_dias_homem', 'producao_l_dia', 
            'ccs_mensal', 'cpp_mensal', 'gordura_mensal', 'proteina_mensal',
            'vacas_lactacao', 'vacas_secas', 'bezerras_aleitamento', 
            'bezerros_aleitamento', 'novilhas', 'reprodutores', 
            'receptoras', 'rebanho_total','valor_pago_produtor','valor_pago_agroindustria'
        ]

        for col in numeric_columns:
            if col in df_visitas.columns:
                df_visitas[col] = pd.to_numeric(df_visitas[col], errors='coerce')

        # Verificar se há registros duplicados
        duplicated_count = df_visitas.duplicated(subset=['id_atendimento']).sum()
        print(f"Registros duplicados por id_atendimento: {duplicated_count}")

        # Remover duplicados
        df_visitas.drop_duplicates(subset=['id_atendimento'], keep='first', inplace=True)

        duplicated_count = df_visitas.duplicated(subset=['id_atendimento']).sum()
        print(f"Registros duplicados pós-tratamento: {duplicated_count}")

        # Corrigir tipos de dados
        df_visitas = corrigir_tipos_dataframe(df_visitas)

        # Voltar ao diretório original
        # os.chdir gerido no setup

        # Calcular tempo de processamento
        fim = datetime.now()
        duracao = (fim - inicio).total_seconds()

        print(f"\n✅ Processamento concluído em {duracao:.2f} segundos")
        print(f"✅ Total de registros no DataFrame final: {len(df_visitas)}")

        return df_visitas

    except Exception as e:
        print(f"❌ Erro durante o processamento dos relatórios: {str(e)}")
        traceback.print_exc()

        # Voltar ao diretório original
        # os.chdir gerido no setup

        return None

def corrigir_tipos_dataframe(df):
    """
    Corrige os tipos de dados do DataFrame para compatibilidade com o Supabase

    Args:
        df: DataFrame a ser corrigido

    Returns:
        DataFrame com tipos corrigidos
    """
    # Cópia para não modificar o original
    df_corrigido = df.copy()

    # Colunas que devem ser inteiros
    colunas_inteiras = [
        'vacas_lactacao', 
        'vacas_secas', 
        'bezerras_aleitamento', 
        'bezerros_aleitamento', 
        'novilhas', 
        'reprodutores', 
        'receptoras', 
        'rebanho_total'
    ]

    # Converter colunas para inteiro
    for col in colunas_inteiras:
        if col in df_corrigido.columns:
            # Converter para float primeiro para lidar com NaN, depois para int
            df_corrigido[col] = pd.to_numeric(df_corrigido[col], errors='coerce')
            df_corrigido[col] = df_corrigido[col].fillna(0)
            df_corrigido[col] = df_corrigido[col].astype(int)

    # Colunas que devem ser float
    colunas_float = [
        'area_pecuaria_ha', 
        'mdo_dias_homem', 
        'producao_l_dia', 
        'ccs_mensal', 
        'cpp_mensal', 
        'gordura_mensal', 
        'proteina_mensal',
        'valor_pago_produtor',
        'valor_pago_agroindustria'
    ]

    # Converter colunas para float
    for col in colunas_float:
        if col in df_corrigido.columns:
            df_corrigido[col] = pd.to_numeric(df_corrigido[col], errors='coerce')
            df_corrigido[col] = df_corrigido[col].fillna(0)

    # Garantir que a coluna data_visita seja datetime
    if 'data_visita' in df_corrigido.columns:
        df_corrigido['data_visita'] = pd.to_datetime(df_corrigido['data_visita'], errors='coerce')

    # Garantir que id_atendimento seja string
    if 'id_atendimento' in df_corrigido.columns:
        df_corrigido['id_atendimento'] = df_corrigido['id_atendimento'].astype(str)

    return df_corrigido

def criar_id_composto(row):
    """
    Cria um ID composto usando MD5 hash de campos-chave

    Args:
        row: Linha do DataFrame

    Returns:
        String com o hash MD5
    """
    # Lista de colunas para compor o ID
    colunas_id = [
        'id_atendimento', 'nome_consultor', 'codigo_lr', 'id_farm', 'nome_produtor', 
        'nome_propriedade', 'data_visita', 'area_pecuaria_ha', 'mdo_dias_homem',
        'producao_l_dia', 'vacas_lactacao', 'vacas_secas', 'bezerras_aleitamento',
        'bezerros_aleitamento', 'novilhas', 'reprodutores', 'receptoras',
        'rebanho_total', 'ccs_mensal', 'cpp_mensal', 'gordura_mensal',
        'proteina_mensal', 'origem_dados','valor_pago_produtor','valor_pago_agroindustria'
    ]

    # Concatenar valores das colunas
    valores = []
    for col in colunas_id:
        if col in row.index:
            valor = row[col]
            # Converter para string e tratar valores nulos
            if pd.isna(valor):
                if col in ['area_pecuaria_ha', 'mdo_dias_homem', 'producao_l_dia', 
                          'vacas_lactacao', 'vacas_secas', 'bezerras_aleitamento',
                          'bezerros_aleitamento', 'novilhas', 'reprodutores', 
                          'receptoras', 'rebanho_total', 'ccs_mensal', 'cpp_mensal', 
                          'gordura_mensal', 'proteina_mensal','valor_pago_produtor','valor_pago_agroindutria']:
                    valores.append('0')
                else:
                    valores.append('')
            else:
                valores.append(str(valor).lower().strip())
        else:
            valores.append('')

    # Juntar valores e criar hash MD5
    texto_composto = '|'.join(valores)
    return hashlib.md5(texto_composto.encode('utf-8')).hexdigest()

def etl_visitas(df_visitas=None, diretorio_dados=None, DIRETORIO_ENV=DIRETORIO_ENV):
    """
    Função ETL para atualizar a tabela de visitas no Supabase

    Args:
        df_visitas: DataFrame já tratado com os dados de visitas (opcional)
        diretorio_dados: Diretório onde estão os arquivos de relatórios (opcional)
        DIRETORIO_ENV: Diretório onde está o arquivo .env
    
    Returns:
        bool: True se o processo foi concluído com sucesso, False caso contrário
    """
    try:
        print("=== ETL DE VISITAS ===")
        inicio_total = datetime.now()

        # Se o DataFrame não foi fornecido, processar os relatórios
        if df_visitas is None:
            if diretorio_dados is None:
                print("❌ É necessário fornecer o DataFrame ou o diretório dos dados.")
                return False

            print("🔍 DataFrame não fornecido. Processando relatórios...")
            df_visitas = processar_relatorios_visitas(diretorio_dados)

            if df_visitas is None:
                print("❌ Falha ao processar os relatórios de visitas.")
                return False

        # Mudar para o diretório onde está o arquivo .env
        # os.chdir gerido no setup
        print(f"Diretório alterado para: {DIRETORIO_ENV}")

        # ETAPA 1: CRIAR ID COMPOSTO E PREPARAR DADOS
        print("\n🔄 ETAPA 1: CRIANDO ID COMPOSTO E PREPARANDO DADOS")

        # Converter data_visita para datetime se ainda não for
        if 'data_visita' in df_visitas.columns and not pd.api.types.is_datetime64_any_dtype(df_visitas['data_visita']):
            df_visitas['data_visita'] = pd.to_datetime(df_visitas['data_visita'], errors='coerce')

        # Adicionar data_processamento atual
        df_visitas['data_processamento'] = datetime.now()

        # Criar coluna de ID composto após ajustar os tipos de dados
        print("🔄 Gerando ID composto para cada registro...")
        df_visitas['id_composto'] = df_visitas.apply(criar_id_composto, axis=1)
        print(f"✅ ID composto gerado para {len(df_visitas)} registros")

        # ETAPA 2: BUSCAR REGISTROS EXISTENTES NO SUPABASE
        print("\n🔍 ETAPA 2: BUSCANDO REGISTROS EXISTENTES NO SUPABASE")

        # Carregar credenciais do Supabase
        key = os.getenv('SUPABASE_SERVICE_KEY')
        url = os.getenv('SUPABASE_URL')

        if not key or not url:
            print("❌ Credenciais não encontradas!")
            # os.chdir gerido no setup
            return False

        # Inicializar cliente Supabase
        supabase = create_client(url, key)

        # Buscar dados existentes para comparação
        print("🔍 Buscando registros existentes...")
        
        try:
            # Buscar apenas os id_composto existentes
            resultado = supabase.table(TAB_VISITAS_STAGING).select("id_composto,data_processamento").execute()

            if resultado.data:
                # Obter conjunto de IDs existentes
                ids_existentes = set(item['id_composto'] for item in resultado.data if 'id_composto' in item)
                
                # Obter a data de processamento mais recente
                datas_processamento = [pd.to_datetime(item['data_processamento']) 
                                      for item in resultado.data 
                                      if 'data_processamento' in item and item['data_processamento'] is not None]

                if datas_processamento:
                    ultima_data_processamento = max(datas_processamento).normalize()
                    print(f"✅ Data de processamento mais recente: {ultima_data_processamento}")
                else:
                    ultima_data_processamento = pd.to_datetime(PERIODO_CHECAGEM_INICIO)
                    print("⚠️ Nenhuma data de processamento encontrada. Usando data padrão.")

                print(f"✅ Encontrados {len(ids_existentes)} registros existentes no Supabase")
            else:
                ids_existentes = set()
                ultima_data_processamento = pd.to_datetime(PERIODO_CHECAGEM_INICIO)
                print("⚠️ Nenhum registro encontrado no Supabase.")
        except Exception as e:
            print(f"⚠️ Erro ao buscar registros existentes: {str(e)}")
            # Em caso de erro, assumir que não há registros
            ids_existentes = set()
            ultima_data_processamento = pd.to_datetime(PERIODO_CHECAGEM_INICIO)

        # ETAPA 3: IDENTIFICAR REGISTROS NOVOS OU ALTERADOS
        print("\n🔍 ETAPA 3: IDENTIFICANDO REGISTROS NOVOS OU ALTERADOS")

        # Identificar registros novos ou alterados usando ID composto
        ids_atuais = set(df_visitas['id_composto'])

        # Registros novos (não existem no Supabase)
        ids_novos = ids_atuais - ids_existentes
        df_novos = df_visitas[df_visitas['id_composto'].isin(ids_novos)]

        # Registros a atualizar (existem no Supabase)
        ids_atualizar = ids_atuais.intersection(ids_existentes)
        df_atualizar = df_visitas[df_visitas['id_composto'].isin(ids_atualizar)]

        print(f"✅ Registros novos por ID composto: {len(df_novos)}")
        print(f"✅ Registros a atualizar: {len(df_atualizar)}")

        # ETAPA 4: INSERIR NO SUPABASE
        print("\n🔄 ETAPA 4: INSERINDO REGISTROS NO SUPABASE")

        # Preparar para inserção
        df_prep = df_novos.copy()  # Apenas registros novos, já que id_composto é a chave primária

        # Converter datas para formato ISO
        for col in ['data_visita', 'data_processamento']:
            if col in df_prep.columns and pd.api.types.is_datetime64_any_dtype(df_prep[col]):
                df_prep[col] = df_prep[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

        # Converter NaN para None
        df_prep = df_prep.replace({np.nan: None})

        # Converter para lista de dicionários
        registros = df_prep.to_dict(orient='records')

        # Definir tamanho do lote
        lote = 100
        total = len(registros)

        # Se não houver registros para inserir, mostrar mensagem e encerrar
        if total == 0:
            print("✅ Nenhum registro novo para inserir.")

            # Mostrar resumo
            fim = datetime.now()
            duracao_total = (fim - inicio_total).total_seconds()

            print("\n=== RESUMO DO ETL DE VISITAS ===")
            print(f"📊 Total de registros no DataFrame: {len(df_visitas)}")
            print(f"🆕 Registros novos: {len(df_novos)}")
            print(f"🔄 Registros já existentes: {len(df_atualizar)}")
            print(f"✅ Nenhum registro novo para inserir.")
            print(f"⏱️ Tempo total do ETL: {duracao_total:.2f} segundos")
            print(f"📅 Data/hora de conclusão: {fim.strftime('%Y-%m-%d %H:%M:%S')}")

            # Voltar ao diretório original
            # os.chdir gerido no setup

            return True

        total_lotes = (total + lote - 1) // lote

        # Contadores
        sucesso = 0
        erro = 0

        inicio = datetime.now()

        # Processar em lotes
        for i in range(0, total, lote):
            lote_atual = registros[i:i+lote]
            num_lote = i // lote + 1

            try:
                print(f"Processando lote {num_lote}/{total_lotes} ({len(lote_atual)} registros)...")

                # Realizar insert simples (não precisamos de upsert já que id_composto é a chave primária)
                resultado = supabase.table(TAB_VISITAS_STAGING).insert(lote_atual).execute()
                
                # Verificar resultado
                if hasattr(resultado, 'error') and resultado.error:
                    print(f"❌ Erro no lote {num_lote}: {resultado.error}")
                    erro += len(lote_atual)
                else:
                    sucesso += len(lote_atual)
                    print(f"✅ Lote {num_lote} processado com sucesso.")

                # Pausa para não sobrecarregar a API
                if num_lote < total_lotes:
                    time.sleep(0.5)

            except Exception as e:
                print(f"❌ Erro no lote {num_lote}: {str(e)}")
                erro += len(lote_atual)

        # Mostrar resumo
        fim = datetime.now()
        duracao = (fim - inicio).total_seconds()
        duracao_total = (fim - inicio_total).total_seconds()

        print("\n=== RESUMO DO ETL DE VISITAS ===")
        print(f"📊 Total de registros no DataFrame: {len(df_visitas)}")
        print(f"🆕 Registros novos: {len(df_novos)}")
        print(f"🔄 Registros já existentes: {len(df_atualizar)}")
        print(f"✅ Registros processados com sucesso: {sucesso}")
        print(f"❌ Registros com erro: {erro}")
        print(f"⏱️ Tempo de inserção: {duracao:.2f} segundos")
        print(f"⏱️ Tempo total do ETL: {duracao_total:.2f} segundos")
        print(f"📅 Data/hora de conclusão: {fim.strftime('%Y-%m-%d %H:%M:%S')}")

        # Voltar ao diretório original
        # os.chdir gerido no setup

        return True

    except Exception as e:
        print(f"❌ Erro durante a execução: {str(e)}")
        traceback.print_exc()

        # Voltar ao diretório original
        # os.chdir gerido no setup

        return False


## Execução: Consolidação e Atualização da Tabela de Visitas (tab_visitas_sq)


In [ ]:
# 1. Consolidar as visitas dos 5 relatórios Excel
df_visitas_consolidado = processar_relatorios_visitas(DIR_BD_SQ)

# 2. Atualizar a tabela de staging de visitas no Supabase
if df_visitas_consolidado is not None and not df_visitas_consolidado.empty:
    resultado_tab_visitas = etl_visitas(df_visitas_consolidado)
    if resultado_tab_visitas:
        print('✅ ETL de visitas concluído com sucesso no Supabase!')
    else:
        print('❌ Falha ao atualizar a tabela de visitas no Supabase!')
else:
    print('❌ Falha ao consolidar relatórios de visitas!')


# 6. Análise de Indicadores de Visitas


In [ ]:

# Diretório onde está o arquivo .env
DIRETORIO_ENV = DIR_BD_SQ
# os.chdir gerido no setup

# Carregar variáveis de ambiente

# Configurar conexão com o Supabase
supabase_url = os.getenv('SUPABASE_URL')
supabase_key = os.getenv('SUPABASE_SERVICE_KEY')
supabase = create_client(supabase_url, supabase_key)

print("=== CRIANDO TABELA FATO DE VISITAS ===")

# 1. Obter dados de visitas
print("\n🔍 ETAPA 1: IMPORTANDO DADOS DE VISITAS")

# Definir período de consulta
data_inicial = DATA_INICIAL_ELABORE
data_final = datetime.now().strftime('%Y-%m-%d')

# Definir colunas a serem importadas
colunas_visitas = 'id_atendimento,nome_consultor,codigo_lr,id_farm,nome_produtor,data_visita,origem_dados'

try:
    # Executar a consulta com filtros de data
    resultado_visitas = supabase.table(TAB_VISITAS_STAGING) \
        .select(colunas_visitas) \
        .gte('data_visita', data_inicial) \
        .lte('data_visita', data_final) \
        .execute()
    
    if resultado_visitas.data:
        df_visitas = pd.DataFrame(resultado_visitas.data)

        # Converter a coluna de data para o tipo datetime
        df_visitas['data_visita'] = pd.to_datetime(df_visitas['data_visita'])
        # Mostrar a data mais recente
        print(f"Data mais recente no relatório de visitas: {df_visitas.data_visita.max()}")

        # Adicionar coluna mes_ano para facilitar o merge
        df_visitas['mes_ano'] = df_visitas['data_visita'].dt.strftime('%Y-%m')
        # Mostrar a data mais recente
        print(f"Mes-Ano mais recente no relatório de visitas: {df_visitas.mes_ano.max()}")
        print(f"✅ Importação de visitas concluída: {len(df_visitas)} registros")
    else:
        print("⚠️ Nenhum registro de visita encontrado para o período especificado.")
        df_visitas = pd.DataFrame()

except Exception as e:
    print(f"❌ Erro ao importar dados de visitas: {str(e)}")
    df_visitas = pd.DataFrame()

# 2. Obter dados de produtores ativos
print("\n🔍 ETAPA 2: IMPORTANDO DADOS DE PRODUTORES ATIVOS")

projetos = ['REGENERA', 'ALVOAR ECO', 'ALVOAR ASSIST', 'SEMEAR', 'ATEG_CCPR', 'Alvoar', 'LPA']  # Lista de projetos a filtrar

try:
    # Executar a consulta sem filtro em coluna inexistente (data_referencia)
    resultado_vinculos = supabase.table(TAB_VINCULOS_STAGING) \
        .select('*') \
        .in_('projeto', projetos) \
        .execute()

    if resultado_vinculos.data:
        df_vinculos_mes = pd.DataFrame(resultado_vinculos.data)
        if 'consultor_grupo_atendimento' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
            df_vinculos_mes.rename(columns={'consultor_grupo_atendimento': 'nome_consultor'}, inplace=True)
        elif 'consultor' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
            df_vinculos_mes.rename(columns={'consultor': 'nome_consultor'}, inplace=True)

        if 'data_referencia' not in df_vinculos_mes.columns:
            df_vinculos_mes['data_referencia'] = pd.to_datetime(
                df_vinculos_mes.get('data_processamento', df_vinculos_mes.get('data_associacao', pd.Timestamp.now()))
            )
        else:
            df_vinculos_mes['data_referencia'] = pd.to_datetime(df_vinculos_mes['data_referencia'])

        # Retirar os grupos de atendimento do CFT
        grupo_cft = [
            'DAYANNE UCHOA VEIGA / DEBORA LIMA DE OLIVEIRA / MARIO BARBOSA ROSA FILHO / MATEUS CARNIELLI / TALITA FONTES / THAYNAN FERREIRA DE ARAUJO',
            'HUGO LOPES / MATEUS CARNIELLI / ROMARCIO PAULO DE OLIVEIRA / THAYNAN FERREIRA DE ARAUJO',
            'BRUNO ANTONIO FERRONI RODRIGUES / HUGO LOPES / MATEUS CARNIELLI / THAYNAN FERREIRA DE ARAUJO',
            'MATHEUS GOMIDES GONCALVES', 'TALITA FONTES'
        ]
        if 'nome_consultor' in df_vinculos_mes.columns:
            df_vinculos_mes = df_vinculos_mes[~df_vinculos_mes['nome_consultor'].isin(grupo_cft)]

        # Adicionar coluna mes_ano para facilitar o merge
        df_vinculos_mes['mes_ano'] = df_vinculos_mes['data_referencia'].dt.strftime('%Y-%m')

        print(f"Data mais recente no relatório de vínculos: {df_vinculos_mes.data_referencia.max() if not df_vinculos_mes.empty else 'N/A'}")
        print(f"Mes-Ano mais recente no relatório de vínculos: {df_vinculos_mes.mes_ano.max() if not df_vinculos_mes.empty else 'N/A'}")
        print(f"✅ Importação de produtores ativos concluída: {len(df_vinculos_mes)} registros")
    else:
        print("⚠️ Nenhum registro de produtor ativo encontrado para o período especificado.")
        df_vinculos_mes = pd.DataFrame()

except Exception as e:
    print(f"❌ Erro ao importar dados de produtores ativos: {str(e)}")
    df_vinculos_mes = pd.DataFrame()

# 3. Preparar os dados para o merge
print("\n🔄 ETAPA 3: PREPARANDO DADOS PARA INTEGRAÇÃO")

# Normalizar nomes de colunas para o merge
df_vinculos_mes = df_vinculos_mes.rename(columns={
    'consultor': 'nome_consultor',
    'projeto': 'origem_dados'
})

# Verificar se as colunas necessárias existem
colunas_necessarias_visitas = ['codigo_lr', 'nome_consultor', 'mes_ano']
colunas_necessarias_vinculos = ['codigo_lr', 'nome_consultor', 'mes_ano']

for coluna in colunas_necessarias_visitas:
    if coluna not in df_visitas.columns:
        print(f"❌ Coluna {coluna} não encontrada em df_visitas")
        exit()

for coluna in colunas_necessarias_vinculos:
    if coluna not in df_vinculos_mes.columns:
        print(f"❌ Coluna {coluna} não encontrada em df_vinculos_mes")
        exit()

# 4. Criar a tabela fato PRINCIPAL (AGREGADA)
print("\n🔄 ETAPA 4: CRIANDO TABELA FATO PRINCIPAL (AGREGADA)")

# Agrupar visitas por produtor, consultor e mês para contar o número de visitas
# Esta é a contagem CORRETA de visitas para cada combinação (produtor, consultor, mês)
visitas_agrupadas = df_visitas.groupby(['codigo_lr', 'nome_consultor', 'mes_ano']).size().reset_index(name='qtd_visitas')

# Criar uma tabela base com todas as combinações de produtor-consultor-mês dos vínculos
# Esta será a granularidade da nossa tabela fato principal
df_fato_base = df_vinculos_mes[['codigo_lr', 'nome_consultor', 'nome_produtor', 'nome_propriedade',
                                 'origem_dados', 'unidade_atendimento', 'cidade_produtor',
                                 'estado_produtor', 'mes_ano']].copy()

# Fazer o merge com as visitas agrupadas (left join para manter todos os produtores ativos)
# O resultado df_fato_principal terá uma linha por (codigo_lr, nome_consultor, mes_ano)
df_fato_principal = pd.merge(
    df_fato_base,
    visitas_agrupadas,
    on=['codigo_lr', 'nome_consultor', 'mes_ano'],
    how='left'
)

# Preencher valores nulos na coluna de quantidade de visitas com 0 (para produtores sem visita)
df_fato_principal['qtd_visitas'] = df_fato_principal['qtd_visitas'].fillna(0)

# Adicionar coluna binária para indicar se houve visita
df_fato_principal['visitado'] = np.where(df_fato_principal['qtd_visitas'] > 0, 1, 0)

# A df_fato_principal agora é a sua df_fato_final para os cálculos de métricas
df_fato_final = df_fato_principal.copy() # Renomeando para manter a consistência com o restante do seu código

#  NOVO: Criar a tabela de detalhes de visitas (para o drill-down)
print("\n🔄 ETAPA 5: CRIANDO TABELA DE DETALHES DE VISITAS")

# Selecionar as colunas relevantes do df_visitas original
df_visitas_detalhe = df_visitas[['id_atendimento', 'data_visita', 'codigo_lr', 'id_farm', 'nome_consultor', 'mes_ano', 'nome_produtor']].copy()

# Opcional: Adicionar informações do vínculo para enriquecer a tabela de detalhes
# Isso pode ser útil se você quiser exibir mais detalhes do produtor/propriedade
df_visitas_detalhe = pd.merge(
    df_visitas_detalhe,
    df_vinculos_mes[['codigo_lr', 'nome_produtor', 'nome_propriedade', 'unidade_atendimento', 'cidade_produtor', 'estado_produtor', 'origem_dados', 'mes_ano']],
    on=['codigo_lr', 'nome_produtor', 'mes_ano'], # Use nome_produtor para garantir que o merge seja correto se houver múltiplos consultores para o mesmo produtor
    how='left',
    suffixes=('_visita', '_vinculo')
)

# Tratar colunas duplicadas após o merge, se necessário
# Por exemplo, se 'nome_produtor_visita' e 'nome_produtor_vinculo' são iguais, manter apenas um
for col in ['nome_produtor', 'origem_dados']:
    if f'{col}_visita' in df_visitas_detalhe.columns and f'{col}_vinculo' in df_visitas_detalhe.columns:
        df_visitas_detalhe[col] = df_visitas_detalhe[f'{col}_visita'].fillna(df_visitas_detalhe[f'{col}_vinculo'])
        df_visitas_detalhe.drop(columns=[f'{col}_visita', f'{col}_vinculo'], inplace=True)
    elif f'{col}_vinculo' in df_visitas_detalhe.columns:
        df_visitas_detalhe[col] = df_visitas_detalhe[f'{col}_vinculo']
        df_visitas_detalhe.drop(columns=[f'{col}_vinculo'], inplace=True)
    elif f'{col}_visita' in df_visitas_detalhe.columns:
        df_visitas_detalhe[col] = df_visitas_detalhe[f'{col}_visita']
        df_visitas_detalhe.drop(columns=[f'{col}_visita'], inplace=True)


print(f"✅ Tabela de detalhes de visitas criada: {len(df_visitas_detalhe)} registros")


# 6. Finalizar e salvar os resultados
print("\n✅ ETAPA 6: FINALIZANDO E SALVANDO RESULTADOS")

# Ordenar o DataFrame principal
df_fato_final = df_fato_final.loc[:, ~df_fato_final.columns.duplicated()].copy()
df_fato_final = df_fato_final.sort_values(by=['mes_ano', 'origem_dados', 'nome_consultor', 'codigo_lr'])
df_fato_final = df_fato_final.loc[(df_fato_final['unidade_atendimento']!='UNIDADE GENERICA') & (df_fato_final['nome_consultor']!='TALITA FONTES')]

# Adicionar coluna de data de processamento
df_fato_final['data_processamento'] = datetime.now()

# Mostrar a data mais recente
print(f"Mes-Ano mais recente da df_fato_final: {df_fato_final.mes_ano.max()}")

# Exibir informações sobre a tabela fato
print(f"\n✅ Tabela fato principal criada com sucesso!")
print(f"Total de registros: {len(df_fato_final)}")

# Exibir informações sobre a tabela de detalhes
print(f"\n✅ Tabela de detalhes de visitas criada com sucesso!")
print(f"Total de registros: {len(df_visitas_detalhe)}")

# 7. Tabela Fato: Visitas (f_Visitas)


In [ ]:

# Diretório onde está o arquivo .env
DIRETORIO_ENV = DIR_BD_SQ
# os.chdir gerido no setup

# Carregar variáveis de ambiente

# Configurar conexão com o Supabase
supabase_url = os.getenv('SUPABASE_URL')
supabase_key = os.getenv('SUPABASE_SERVICE_KEY')
supabase = create_client(supabase_url, supabase_key)

# Nome da tabela no Supabase
SUPABASE_TABLE_NAME = TAB_VISITAS_FATO

def run_etl_and_upsert_f_visitas(data_inicial: str, data_final: str) -> pd.DataFrame:
    """
    Executa o processo ETL para construir a tabela f_visitas e realiza um upsert
    (delete/insert granular) no Supabase para o período especificado,
    sem depender da coluna 'id_composto' no Supabase.

    Args:
        data_inicial (str): Data de início do período de consulta (YYYY-MM-DD).
        data_final (str): Data de fim do período de consulta (YYYY-MM-DD).

    Returns:
        pd.DataFrame: O DataFrame f_visitas final após o tratamento e antes do upsert.
    """
    print("=== INICIANDO ROTINA ETL E UPSERT GRANULAR PARA F_VISITAS ===")

    # 1. Obter dados de visitas
    print("\n🔍 ETAPA 1: IMPORTANDO DADOS DE VISITAS")
    colunas_visitas = 'id_atendimento,nome_consultor,codigo_lr,id_farm,nome_produtor,data_visita,origem_dados,valor_pago_produtor,valor_pago_agroindustria'
    try:
        resultado_visitas = supabase.table(TAB_VISITAS_STAGING) \
            .select(colunas_visitas) \
            .gte('data_visita', data_inicial) \
            .lte('data_visita', data_final) \
            .execute()
        if resultado_visitas.data:
            df_visitas = pd.DataFrame(resultado_visitas.data)
            df_visitas['data_visita'] = pd.to_datetime(df_visitas['data_visita'])
            df_visitas['mes_ano'] = df_visitas['data_visita'].dt.strftime('%Y-%m')
            df_visitas['mes_referencia'] = df_visitas['data_visita'].dt.to_period('M').dt.to_timestamp()
            df_visitas['valor_pago_produtor'] = pd.to_numeric(df_visitas['valor_pago_produtor'], errors='coerce').astype('Float64')
            df_visitas['valor_pago_agroindustria'] = pd.to_numeric(df_visitas['valor_pago_agroindustria'], errors='coerce').astype('Float64')
            print(f"✅ Importação de visitas concluída: {len(df_visitas)} registros")
        else:
            print("⚠️ Nenhum registro de visita encontrado para o período especificado.")
            df_visitas = pd.DataFrame()
    except Exception as e:
        print(f"❌ Erro ao importar dados de visitas: {str(e)}")
        df_visitas = pd.DataFrame()

        # 2. Obter dados de produtores ativos
    print("🔍 ETAPA 2: IMPORTANDO DADOS DE PRODUTORES ATIVOS")
    projetos = ['REGENERA', 'ALVOAR ECO', 'ALVOAR ASSIST', 'SEMEAR', 'ATEG_CCPR', 'Alvoar', 'LPA']
    try:
        resultado_vinculos = supabase.table(TAB_VINCULOS_STAGING).select('*').in_('projeto', projetos).execute()
        if resultado_vinculos.data:
            df_vinculos_mes = pd.DataFrame(resultado_vinculos.data)
            if 'consultor_grupo_atendimento' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
                df_vinculos_mes.rename(columns={'consultor_grupo_atendimento': 'nome_consultor'}, inplace=True)
            elif 'consultor' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
                df_vinculos_mes.rename(columns={'consultor': 'nome_consultor'}, inplace=True)

            if 'data_referencia' not in df_vinculos_mes.columns:
                df_vinculos_mes['data_referencia'] = pd.to_datetime(df_vinculos_mes.get('data_processamento', df_vinculos_mes.get('data_associacao', pd.Timestamp.now())))
            else:
                df_vinculos_mes['data_referencia'] = pd.to_datetime(df_vinculos_mes['data_referencia'])
                
            df_vinculos_mes['mes_ano'] = df_vinculos_mes['data_referencia'].dt.strftime('%Y-%m')
            df_vinculos_mes['mes_referencia'] = df_vinculos_mes['data_referencia'].dt.to_period('M').dt.to_timestamp()
            
            if 'meses_ativos_vinculo' not in df_vinculos_mes.columns:
                df_vinculos_mes['meses_ativos_vinculo'] = 1
            else:
                df_vinculos_mes['meses_ativos_vinculo'] = pd.to_numeric(df_vinculos_mes['meses_ativos_vinculo'], errors='coerce').fillna(1).astype('Int64')

            if 'codigo_fazenda' not in df_vinculos_mes.columns:
                df_vinculos_mes['codigo_fazenda'] = None

            grupo_cft = [
                'DAYANNE UCHOA VEIGA / DEBORA LIMA DE OLIVEIRA / MARIO BARBOSA ROSA FILHO / MATEUS CARNIELLI / TALITA FONTES / THAYNAN FERREIRA DE ARAUJO',
                'HUGO LOPES / MATEUS CARNIELLI / ROMARCIO PAULO DE OLIVEIRA / THAYNAN FERREIRA DE ARAUJO',
                'BRUNO ANTONIO FERRONI RODRIGUES / HUGO LOPES / MATEUS CARNIELLI / THAYNAN FERREIRA DE ARAUJO',
                'MATHEUS GOMIDES GONCALVES'
            ]
            if 'nome_consultor' in df_vinculos_mes.columns:
                df_vinculos_mes = df_vinculos_mes[~df_vinculos_mes['nome_consultor'].isin(grupo_cft)]

            print(f"✅ Importação de vínculos concluída: {len(df_vinculos_mes)} registros")
        else:
            print("⚠️ Nenhum registro de vínculo encontrado para o período especificado.")
            df_vinculos_mes = pd.DataFrame()
    except Exception as e:
        print(f"❌ Erro ao importar dados de vínculos: {str(e)}")
        df_vinculos_mes = pd.DataFrame()

    # 3. Validação e Preparação dos DataFrames
    print("\n🔍 ETAPA 3: VALIDANDO E PREPARANDO DATAFRAMES")
    if df_visitas.empty or df_vinculos_mes.empty:
        print("❌ Não foi possível criar a tabela fato devido a DataFrames vazios.")
        return pd.DataFrame()

    colunas_necessarias_visitas = ['id_atendimento', 'nome_consultor', 'codigo_lr', 'nome_produtor', 'data_visita', 'mes_ano', 'mes_referencia', 'origem_dados','valor_pago_produtor','valor_pago_agroindustria']
    colunas_necessarias_vinculos = ['codigo_lr', 'nome_consultor', 'mes_ano', 'mes_referencia', 'nome_produtor', 'nome_propriedade', 'projeto', 'unidade_atendimento', 'cidade_produtor', 'estado_produtor', 'codigo_agroindustria','codigo_fazenda','meses_ativos_vinculo']

    for coluna in colunas_necessarias_visitas:
        if coluna not in df_visitas.columns:
            print(f"❌ Coluna {coluna} não encontrada em df_visitas")
            return pd.DataFrame()
    for coluna in colunas_necessarias_vinculos:
        if coluna not in df_vinculos_mes.columns:
            print(f"❌ Coluna {coluna} não encontrada em df_vinculos_mes")
            return pd.DataFrame()

    # 4. Construindo a Tabela Fato F_VISITAS
    print("\n🔄 ETAPA 4: CONSTRUINDO A TABELA FATO F_VISITAS")
    df_base_vinculos = df_vinculos_mes[[
        'codigo_lr', 'nome_consultor', 'mes_ano', 'mes_referencia','meses_ativos_vinculo', 'unidade_atendimento', 'nome_produtor', 'nome_propriedade',
        'projeto', 'codigo_agroindustria', 'codigo_fazenda', 'cidade_produtor', 'estado_produtor'
    ]].drop_duplicates(subset=['codigo_lr', 'nome_consultor', 'mes_ano']).copy()

    df_visitas_para_merge = df_visitas[[
        'id_atendimento', 'data_visita', 'codigo_lr','nome_consultor', 'mes_ano','valor_pago_produtor','valor_pago_agroindustria'
    ]].copy()
    
    # Normalizar chaves de junção para garantir casamento perfeito
    # Normalizar chaves e remover caracteres invisiveis (\xa0)
    df_base_vinculos['codigo_lr'] = df_base_vinculos['codigo_lr'].astype(str).str.strip().str.replace('\xa0', '').str.upper()
    df_visitas_para_merge['codigo_lr'] = df_visitas_para_merge['codigo_lr'].astype(str).str.strip().str.replace('\xa0', '').str.upper()
    df_visitas_para_merge = df_visitas_para_merge.drop(columns=['nome_consultor'], errors='ignore')

    f_visitas = pd.merge(
        df_base_vinculos,
        df_visitas_para_merge,
        on=['codigo_lr', 'mes_ano'],
        how='left'
    )

    # Antes: o id_farm vinha da tab-visitas. Agora, vem da tab_vinculos Renomear para não mudar estrutura
    # Garantir que f_visitas armazene APENAS eventos de visitas efetivamente realizadas (com data_visita válida)
    f_visitas = f_visitas[f_visitas['data_visita'].notna()].copy()

    f_visitas.rename(columns={'codigo_fazenda':'id_farm'},inplace=True)
    
    # 5. Finalizar e preparar para o Supabase
    print("\n✅ ETAPA 5: FINALIZANDO E PREPARANDO PARA SUPABASE")
    f_visitas = f_visitas.loc[(f_visitas['unidade_atendimento'] != 'UNIDADE GENERICA') & (f_visitas['nome_consultor'] != 'TALITA FONTES')]
    f_visitas.drop(columns=['mes_ano', 'unidade_atendimento', 'cidade_produtor', 'estado_produtor'], inplace=True)

    # Converter id_atendimento para o tipo Int64 (inteiro que aceita NaN/None)
    f_visitas['id_atendimento'] = f_visitas['id_atendimento'].astype('Int64')

    # Adicionar coluna de data de processamento (datetime)
    # Esta coluna NÃO fará parte do hash para garantir estabilidade
    f_visitas['data_processamento'] = datetime.now()

    # Formatar colunas de data/datetime para o Supabase (ISO 8601 strings com 'Z' para UTC)
    # ESTAS SÃO AS STRINGS QUE SERÃO ENVIADAS PARA O SUPABASE E USADAS NO HASH (se aplicável)
    f_visitas['data_visita_str'] = f_visitas['data_visita'].apply(
        lambda x: x.isoformat(timespec='milliseconds') + 'Z' if pd.notna(x) else None
    )
    f_visitas['mes_referencia_str'] = f_visitas['mes_referencia'].apply(
        lambda x: x.isoformat(timespec='milliseconds') + 'Z' if pd.notna(x) else None
    )
    # data_processamento_str é formatada para envio, mas NÃO para o hash
    f_visitas['data_processamento_str'] = f_visitas['data_processamento'].apply(
        lambda x: x.isoformat(timespec='milliseconds') + 'Z' if pd.notna(x) else None
    )

    # Reordenar o DataFrame final
    f_visitas = f_visitas.sort_values(by=['mes_referencia', 'projeto', 'nome_consultor', 'codigo_lr', 'data_visita'])

    print(f"✅ Tabela f_visitas tratada e pronta para upsert: {len(f_visitas)} registros")
    print("\nExemplo de f_visitas (primeiras 5 linhas com id_composto):")
    print(f_visitas.head())

    #  Geração do id_composto LOCALMENTE para f_visitas
    # Colunas que formam a chave composta para o hash
    hash_cols = ['codigo_lr', 'nome_consultor', 'mes_referencia_str', 'id_atendimento']
    
    f_visitas_temp_for_hash = f_visitas[hash_cols].copy()
    
    # Tratar None/NaN para string 'NULL_VAL' para o hash
    # IMPORTANTE: id_atendimento deve ser tratado como string para o hash
    f_visitas_temp_for_hash['id_atendimento'] = f_visitas_temp_for_hash['id_atendimento'].astype(str).replace({'<NA>': 'NULL_VAL'})
    f_visitas_temp_for_hash['mes_referencia_str'] = f_visitas_temp_for_hash['mes_referencia_str'].astype(str).replace({'None': 'NULL_VAL'})
    # Adicione tratamento para outras colunas em hash_cols se elas puderem ser None/NaN
    f_visitas_temp_for_hash['codigo_lr'] = f_visitas_temp_for_hash['codigo_lr'].astype(str).replace({'None': 'NULL_VAL', 'nan': 'NULL_VAL'})
    f_visitas_temp_for_hash['nome_consultor'] = f_visitas_temp_for_hash['nome_consultor'].astype(str).replace({'None': 'NULL_VAL', 'nan': 'NULL_VAL'})
    
    
    f_visitas_hash_input = f_visitas_temp_for_hash.agg(''.join, axis=1)
    f_visitas['id_composto'] = f_visitas_hash_input.apply(lambda x: hashlib.sha256(x.encode()).hexdigest())
    #  FIM da Geração do id_composto LOCALMENTE


    # 6. Rotina de UPSERT no Supabase
    print("\n🚀 ETAPA 6: REALIZANDO UPSERT NO SUPABASE")

    # Adicionar a coluna 'id_composto' ao DataFrame que será enviado para o Supabase
    # Isso é crucial para que o upsert saiba qual registro usar para o conflito
    f_visitas_to_upsert = f_visitas.copy()
    
    #  REMOVER DUPLICATAS DE id_composto DO DATAFRAME ANTES DO UPSERT
    initial_rows_upsert = len(f_visitas_to_upsert)
    f_visitas_to_upsert.drop_duplicates(subset=['id_composto'], keep='first', inplace=True)
    if len(f_visitas_to_upsert) < initial_rows_upsert:
        print(f"   - Removidas {initial_rows_upsert - len(f_visitas_to_upsert)} linhas duplicadas com base em 'id_composto' antes do UPSERT.")
    
    # Preparar o DataFrame para inserção/upsert, usando as colunas de string de data
    # e removendo as colunas temporárias ou as originais de data/datetime
    f_visitas_to_upsert = f_visitas_to_upsert.drop(columns=['data_visita', 'mes_referencia', 'data_processamento'], errors='ignore')
    f_visitas_to_upsert.rename(columns={
        'data_visita_str': 'data_visita',
        'mes_referencia_str': 'mes_referencia',
        'data_processamento_str': 'data_processamento'
    }, inplace=True)

    # ✅ TRATAMENTO FINAL DE np.nan PARA None EM TODAS AS COLUNAS ANTES DO UPSERT
    print("\n🔄 Tratando np.nan para None em todas as colunas antes do upsert...")
    for col in f_visitas_to_upsert.columns:
        # Para colunas numéricas (Float64, Int64)
        if pd.api.types.is_numeric_dtype(f_visitas_to_upsert[col]):
            if f_visitas_to_upsert[col].isnull().any():
                f_visitas_to_upsert.loc[f_visitas_to_upsert[col].isnull(), col] = None
                # print(f"  ✅ Coluna '{col}': np.nan/NaT transformados para None.") # Opcional: remover para menos logs
        # Para colunas de objeto (string) que podem ter a string 'NaN' ou np.nan
        elif pd.api.types.is_object_dtype(f_visitas_to_upsert[col]):
            if (f_visitas_to_upsert[col] == 'NaN').any():
                f_visitas_to_upsert.loc[f_visitas_to_upsert[col] == 'NaN', col] = None
                # print(f"  ✅ Coluna '{col}': string 'NaN' transformadas para None.") # Opcional: remover para menos logs
            if f_visitas_to_upsert[col].isnull().any():
                f_visitas_to_upsert.loc[f_visitas_to_upsert[col].isnull(), col] = None
                # print(f"  ✅ Coluna '{col}': np.nan em objeto transformados para None.") # Opcional: remover para menos logs
    print("✅ Tratamento de NaN para None concluído.")

    records_to_upsert = f_visitas_to_upsert.to_dict(orient='records')

    print(f"   - Realizando UPSERT de {len(records_to_upsert)} registros no Supabase...")

    chunk_size = 1000
    for i in range(0, len(records_to_upsert), chunk_size):
        chunk = records_to_upsert[i:i + chunk_size]
        try:
            # Usar a função upsert do Supabase, especificando 'id_composto' como a coluna de conflito
            response = supabase.table(SUPABASE_TABLE_NAME) \
                .upsert(chunk, on_conflict='id_composto') \
                .execute()
            if response.data:
                print(f"     - UPSERT de {len(response.data)} registros em chunk {i//chunk_size + 1}.")
            else:
                print(f"     - ATENÇÃO: Nenhum registro UPSERTED para chunk {i//chunk_size + 1} ou erro na resposta. Resposta: {response.status_code} - {response.data}")
        except Exception as e:
            print(f"❌ Erro ao realizar UPSERT no Supabase (chunk {i//chunk_size + 1}): {str(e)}")
            # Opcional: Logar o chunk que falhou para depuração
            # print(f"   Chunk que falhou: {chunk}")

    print(f"   - UPSERT concluído. Total de registros processados: {len(records_to_upsert)}")

    print("\n✅ Rotina ETL e UPSERT concluída com sucesso!")
    return f_visitas 

#  Exemplo de como chamar a função



## Execução: Tabela Fato Visitas (f_Visitas)


In [ ]:
data_inicial_etl = '2026-01-01'
# Trazer o dia atual
hoje = date.today()
# Separa primeiro e último dia
_, ultimo_dia = calendar.monthrange(hoje.year, hoje.month)
# Criar a data do último dia do mês atual
data_ultimo_dia = date(hoje.year, hoje.month, ultimo_dia)
# Define a data final do ETL
data_final_etl = data_ultimo_dia

final_f_visitas_df = run_etl_and_upsert_f_visitas(data_inicial_etl, data_final_etl)

if not final_f_visitas_df.empty:
    print(f"\nDataFrame final retornado pela função (primeiras 5 linhas):\n{final_f_visitas_df.head()}")
    data_export = datetime.now().strftime("%Y_%m_%d_%H%M%S")
    pasta_saida = raiz_projeto / "DB" / "OUTPUT" / "PROCESSED"
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho_export = pasta_saida / f"{data_export}_f_visitas_final_pos_upsert.xlsx"
    exportar_xlsx_formatado(df=final_f_visitas_df, caminho_saida=caminho_export)
    aplicar_estilo_listrado_xlsx(caminho_arquivo=caminho_export, cor_cabecalho="#247B72", cor_texto_cabecalho="#FFFFFF", cor_linha_alternada="#F2F2F2", cor_linha_base="#FFFFFF", primeira_linha_cinza=True)
    print(f"✅ Excel formatado exportado com sucesso em: {caminho_export}")


In [ ]:
final_f_visitas_df

# 8. Tabela Fato: Consistência (f_Consistencia)


In [ ]:
# Diretório onde está o arquivo .env
DIRETORIO_ENV = DIR_BD_SQ
# os.chdir gerido no setup

# Carregar variáveis de ambiente

# Configurar conexão com o Supabase
supabase_url = os.getenv('SUPABASE_URL')
supabase_key = os.getenv('SUPABASE_SERVICE_KEY')
supabase = create_client(supabase_url, supabase_key)

# Definir datas de importação
data_inicial = DATA_INICIAL_ANALISE
data_inicial_elabore = DATA_INICIAL_ELABORE

# Trazer o dia atual
hoje = date.today()
# Separa primeiro e último dia
_, ultimo_dia = calendar.monthrange(hoje.year, hoje.month)
# Criar a data do último dia do mês atual
data_ultimo_dia = date(hoje.year, hoje.month, ultimo_dia)
# Define a data final do ETL
data_final =  data_ultimo_dia

# 2. Obter dados de produtores ativos
print("\n🔍 ETAPA 1: IMPORTANDO DADOS DE PRODUTORES ATIVOS")
projetos = ['REGENERA', 'ALVOAR ECO', 'ALVOAR ASSIST', 'SEMEAR', 'ATEG_CCPR', 'Alvoar', 'LPA']
colunas_produtores_ativos = '*'
try:
    resultado_vinculos = supabase.table(TAB_VINCULOS_STAGING) \
        .select(colunas_produtores_ativos) \
        .in_('projeto', projetos) \
        .execute()
    if resultado_vinculos.data:
        df_vinculos_mes = pd.DataFrame(resultado_vinculos.data)
        if 'consultor_grupo_atendimento' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
            df_vinculos_mes.rename(columns={'consultor_grupo_atendimento': 'nome_consultor'}, inplace=True)
        elif 'consultor' in df_vinculos_mes.columns and 'nome_consultor' not in df_vinculos_mes.columns:
            df_vinculos_mes.rename(columns={'consultor': 'nome_consultor'}, inplace=True)

        if 'data_referencia' not in df_vinculos_mes.columns:
            df_vinculos_mes['data_referencia'] = pd.to_datetime(df_vinculos_mes.get('data_processamento', df_vinculos_mes.get('data_associacao', pd.Timestamp.now())))
        else:
            df_vinculos_mes['data_referencia'] = pd.to_datetime(df_vinculos_mes['data_referencia'])

        grupo_cft = [
            'DAYANNE UCHOA VEIGA / DEBORA LIMA DE OLIVEIRA / MARIO BARBOSA ROSA FILHO / MATEUS CARNIELLI / TALITA FONTES / THAYNAN FERREIRA DE ARAUJO',
            'HUGO LOPES / MATEUS CARNIELLI / ROMARCIO PAULO DE OLIVEIRA / THAYNAN FERREIRA DE ARAUJO',
            'BRUNO ANTONIO FERRONI RODRIGUES / HUGO LOPES / MATEUS CARNIELLI / THAYNAN FERREIRA DE ARAUJO',
            'MATHEUS GOMIDES GONCALVES', 'TALITA FONTES'
        ]
        df_vinculos_mes = df_vinculos_mes[~df_vinculos_mes['nome_consultor'].isin(grupo_cft)]
        df_vinculos_mes['mes_referencia'] = df_vinculos_mes['data_referencia'].dt.to_period('M').dt.to_timestamp()
        print(f"✅ Importação de vínculos concluída: {len(df_vinculos_mes)} registros")
    else:
        print("⚠️ Nenhum registro de vínculo encontrado para o período especificado.")
        df_vinculos_mes = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de vínculos: {str(e)}")
    df_vinculos_mes = pd.DataFrame()

#  NOVA ETAPA 1.5: Importar dados de solicitação de vínculo (tab_vinculos_sq)
print("\n🔍 ETAPA 1.5: IMPORTANDO DATAS DE SOLICITAÇÃO DE VÍNCULO (tab_vinculos_sq)")
colunas_vinculos_sq = '*'
try:
    # Importar todos os registros, sem filtro de data, para garantir que todas as datas de solicitação sejam capturadas
    df_vinculos_solicitacao_supabase = (
        supabase
        .table(TAB_VINCULOS_STAGING)
        .select(colunas_vinculos_sq)
        .in_('projeto', projetos)
        .execute()
    )
    
    if df_vinculos_solicitacao_supabase.data:
        df_vinculos_solicitacao = pd.DataFrame(df_vinculos_solicitacao_supabase.data)
        df_vinculos_solicitacao.rename(columns={'consultor_grupo_atendimento': 'nome_consultor'}, inplace=True)
        df_vinculos_solicitacao = df_vinculos_solicitacao.loc[~df_vinculos_solicitacao['nome_consultor'].isin(grupo_cft)]
        df_vinculos_solicitacao['data_associacao'] = pd.to_datetime(df_vinculos_solicitacao['data_associacao']).dt.to_period('M').dt.to_timestamp()

        # Agrupar para pegar a data de solicitação mais antiga por vínculo único
        df_min_solicitacao = df_vinculos_solicitacao.groupby(['codigo_lr', 'nome_consultor'])['data_associacao'].min().reset_index()
        df_min_solicitacao.rename(columns={'data_associacao': 'data_solicitacao_min'}, inplace=True)

        print(f"✅ Importação de datas de solicitação concluída: {len(df_min_solicitacao)} vínculos únicos.")
        print("\nDataFrame df_min_solicitacao (head):")
        print(df_min_solicitacao.head())
    else:
        print("⚠️ Nenhum registro de solicitação de vínculo encontrado.")
        df_min_solicitacao = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de solicitação de vínculo: {str(e)}")
    df_min_solicitacao = pd.DataFrame()
    

# 2. Importar tabela de consistencia mensal
print("\n🔍 ETAPA 2: IMPORTANDO DADOS DE CONSISTENCIA MENSAL")

try:
    resultado_consistencia = supabase.table('tab_consistencia_mensal') \
        .select('*') \
        .gte('mes_referencia', data_inicial) \
        .lte('mes_referencia', data_final) \
        .execute()
    if resultado_consistencia.data:
        df_consistencia_mes = pd.DataFrame(resultado_consistencia.data)
        df_consistencia_mes['mes_referencia'] = pd.to_datetime(df_consistencia_mes['mes_referencia']).dt.to_period('M').dt.to_timestamp()
        df_consistencia_mes['mes_elabore'] = pd.to_datetime(df_consistencia_mes['mes_elabore']).dt.to_period('M').dt.to_timestamp()

# Adicionar logo após criar df_consistencia_mes (ETAPA 2)
codigos_invalidos = ['Teste', 'Teste Gestor', 'Labor Rural']

df_consistencia_mes = df_consistencia_mes[
    df_consistencia_mes['codigo_lr'].notna() &                          # remove NULL
    ~df_consistencia_mes['codigo_lr'].isin(codigos_invalidos) &         # remove lista exata
    ~df_consistencia_mes['codigo_lr'].str.lower().str.contains('teste', na=False) & # remove variações
    ~df_consistencia_mes['codigo_lr'].str.lower().str.contains('labor', na=False)
]
print(f"✅ Registros inválidos removidos. Restam: {len(df_consistencia_mes)}")

# 3. Importar tabela de consistencia mensal
print("\n🔍 ETAPA 3: IMPORTANDO DADOS DE CONSISTENCIA ANUAL")

try:
    resultado_consistencia = supabase.table('tab_consistencia_anual') \
        .select('*') \
        .gte('mes_referencia', data_inicial) \
        .lte('mes_referencia', data_final) \
        .execute()
    if resultado_consistencia.data:
        df_consistencia_anual = pd.DataFrame(resultado_consistencia.data)
        df_consistencia_anual['mes_referencia'] = pd.to_datetime(df_consistencia_anual['mes_referencia']).dt.to_period('M').dt.to_timestamp()
        df_consistencia_anual['mes_elabore'] = pd.to_datetime(df_consistencia_anual['mes_elabore']).dt.to_period('M').dt.to_timestamp()
        print(f"✅ Importação de consistência anual concluída: {len(df_consistencia_anual)} registros")
    else:
        print("⚠️ Nenhum registro de consistência anual encontrado para o período especificado.")
        df_consistencia_anual = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de consistência anual: {str(e)}")
    df_consistencia_anual = pd.DataFrame()

# Fazer o mesmo para df_consistencia_anual (ETAPA 3)
df_consistencia_anual = df_consistencia_anual[
    df_consistencia_anual['codigo_lr'].notna() &
    ~df_consistencia_anual['codigo_lr'].isin(codigos_invalidos) &
    ~df_consistencia_anual['codigo_lr'].str.lower().str.contains('teste', na=False) &
    ~df_consistencia_anual['codigo_lr'].str.lower().str.contains('labor', na=False)
]
print(f"✅ Registros inválidos removidos da anual. Restam: {len(df_consistencia_anual)}")

#  NOVA ETAPA: Aplicar regra de carência de 3 meses aos VÍNCULOS
print("\n⚙️ ETAPA 4: APLICANDO REGRA DE CARÊNCIA DE 3 MESES AOS VÍNCULOS")

df_vinculos_com_carencia = pd.DataFrame() # Inicializa para garantir que existe

if not df_vinculos_mes.empty:
    # 1. Agrupar por 'codigo_lr' e 'nome_consultor' para encontrar a data mínima de vínculo
    # Esta é a data de referência mais antiga do produtor ativo
    df_min_vinculo_ativo = df_vinculos_mes.groupby(['codigo_lr', 'nome_consultor'])['data_referencia'].min().reset_index()
    df_min_vinculo_ativo.rename(columns={'data_referencia': 'data_referencia_min_ativo'}, inplace=True)

    # 2. Mesclar com df_min_solicitacao para obter a data de solicitação mínima
    if not df_min_solicitacao.empty:
        df_min_vinculo_completo = df_min_vinculo_ativo.merge(
            df_min_solicitacao,
            on=['codigo_lr', 'nome_consultor'],
            how='left'
        )
        # Escolher a data de início do vínculo: a menor entre data_referencia_min_ativo e data_solicitacao_min
        # Se data_solicitacao_min for NaN (não encontrado), usar data_referencia_min_ativo
        df_min_vinculo_completo['data_inicio_vinculo'] = df_min_vinculo_completo.apply(
            lambda row: min(row['data_referencia_min_ativo'], row['data_solicitacao_min'])
                        if pd.notna(row['data_solicitacao_min'])
                        else row['data_referencia_min_ativo'],
            axis=1
        )
    else:
        # Se não houver dados de solicitação, usar apenas a data mínima de atividade
        df_min_vinculo_completo = df_min_vinculo_ativo.copy()
        df_min_vinculo_completo['data_inicio_vinculo'] = df_min_vinculo_completo['data_referencia_min_ativo']

    # 3. Calcular a data de carência (data_inicio_vinculo + 3 meses)
    df_min_vinculo_completo['data_carencia_fim'] = df_min_vinculo_completo['data_inicio_vinculo'] + pd.DateOffset(months=2)

    print(f"✅ Datas de carência calculadas para {len(df_min_vinculo_completo)} vínculos únicos usando data de solicitação.")
    print("\nDataFrame df_min_vinculo_completo (head):")
    print(df_min_vinculo_completo.head())

    print("\n🤝 ETAPA DE CARÊNCIA: MESCLANDO E FILTRANDO VÍNCULOS")

    df_vinculos_com_carencia = df_vinculos_mes.merge(
        df_min_vinculo_completo[['codigo_lr', 'nome_consultor', 'data_carencia_fim', 'data_inicio_vinculo']],
        on=['codigo_lr', 'nome_consultor'],
        how='left'
    )
    print(f"✅ Vínculos mesclados com datas de carência: {len(df_vinculos_com_carencia)} registros.")
    print("\nDataFrame df_vinculos_com_carencia (head):")
    print(df_vinculos_com_carencia.head())
else:
    print("⚠️ DataFrame de produtores ativos está vazio, não é possível aplicar a lógica de carência.")
    df_vinculos_com_carencia = pd.DataFrame() # Garante que o DF existe mesmo vazio

#  NOVA ETAPA 5: Integrar Dados de Consistência Mensal e Anual
    print("\n📊 ETAPA 5: INTEGRANDO DADOS DE CONSISTÊNCIA MENSAL E ANUAL")

    if df_vinculos_com_carencia.empty:
        print("⚠️ DataFrame de vínculos com carência está vazio, não é possível integrar dados de consistência.")
        df_vinculos_com_carencia = pd.DataFrame()

# Merge com consistência mensal
# Selecionar apenas as colunas necessárias de df_consistencia_mes para evitar conflitos
cols_consist = [c for c in ['codigo_lr', 'mes_referencia', 'mes_elabore', 'consistencia_mensal', 'status_code', 'detalhamento_inconsistencia'] if c in df_consistencia_mes.columns]
df_consistencia_mes_merge = df_consistencia_mes[cols_consist].copy()

df_final = df_vinculos_com_carencia.merge(
    df_consistencia_mes_merge,
    on=['codigo_lr', 'mes_referencia'],
    how='left'
)
# Preencher False onde não houve match (ou seja, não tem consistência mensal)
print(f"✅ Consistência mensal integrada. Total de registros: {len(df_final)}")

# Merge com consistência anual
# Selecionar apenas as colunas necessárias de df_consistencia_anual
df_consistencia_anual_merge = df_consistencia_anual[['codigo_lr', 'mes_referencia','consistencia_anual']].copy()

df_final = df_final.merge(
    df_consistencia_anual_merge,
    on=['codigo_lr', 'mes_referencia'],
    how='left'
)

# Retirar produtor teste
df_final = df_final.loc[df_final['codigo_lr']!='PRODUTOR_TESTE']
# Preencher False onde não houve match (ou seja, não tem consistência anual)
print(f"✅ Consistência anual integrada. Total de registros: {len(df_final)}")

#  ETAPA 6: Retirar lista de produtores exceção
print("⬆️ ETAPA 6: IMPORTAR TABELA DE PRODUTORES EXCEÇÃO")
try:
    caminho_excecao = buscar_arquivo_mais_recente(DIR_BD_SQ, "*EXCECAO*.xlsx")
    if caminho_excecao and Path(caminho_excecao).exists():
        df_excecao = pd.read_excel(caminho_excecao)
        df_excecao.columns = ['codigo_lr', 'nome_produtor', 'status']
        df_excecao = df_excecao.loc[df_excecao['status']=='ATIVO']
        df_excecao['excecao'] = 1
        df_final = df_final.merge(df_excecao[['codigo_lr','excecao']], on='codigo_lr', how='left')
    else:
        df_final['excecao'] = 0
except Exception as e:
    print(f"⚠️ Tabela de exceção não encontrada ou não processada: {e}")
    df_final['excecao'] = 0

# Preencher NaN com 0 e converter para tipo inteiro.
if 'excecao' in df_final.columns:
    df_final['excecao'] = df_final['excecao'].fillna(0).astype(int)
    print("✅ Coluna 'excecao' tratada (NaN preenchido com 0 e convertida para int).")

#  ETAPA 7: Inserindo tabela de PRODUTORES 12 MESES
print("\n⬆️ ETAPA 7: IMPORTANTO E MESCLANDO PRODUTORES COM 12 MESES")

# Importar do supabase
try:
    produtores_12meses = supabase.table('tab_lancamentos_produtores') \
        .select('idFazenda,mesReferencia,meses_sequenciais') \
        .gte('mesReferencia', data_inicial_elabore) \
        .lte('mesReferencia', data_final) \
        .execute()
    if produtores_12meses.data:
        df_produtores_12meses = pd.DataFrame(produtores_12meses.data)
        df_produtores_12meses['mesReferencia'] = pd.to_datetime(df_produtores_12meses['mesReferencia']).dt.to_period('M').dt.to_timestamp()
        print(df_produtores_12meses.dtypes)
    else:
        print("⚠️ Nenhum registro de produtores com dados encontrado para o período especificado.")
        df_produtores_12meses = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de produtores com dados: {str(e)}")
    df_produtores_12meses = pd.DataFrame()

# Importar tab_fazenda para identificar codigo_lr por idFazenda
try:
    tab_fazenda = supabase.table('tab_fazenda') \
        .select('id,codAgroindustria,idProdutor,idConsultor') \
        .eq('aprovacao','Aprovado') \
        .eq('Excluido', 0) \
        .execute()
    if tab_fazenda.data:
        d_fazenda = pd.DataFrame(tab_fazenda.data)
        d_fazenda = d_fazenda.loc[d_fazenda['codAgroindustria'].notna()]
        d_fazenda = d_fazenda[['id','codAgroindustria','idProdutor','idConsultor']]
        d_fazenda.columns = ['idFazenda', 'codigo_lr','idProdutor','idConsultor'] # Renomear para padrão utilizado nas outras tabelas
        d_fazenda['idFazenda'] = pd.to_numeric(d_fazenda['idFazenda'])
        # Dividir a string por ';' e expandir em novas linhas
        d_fazenda = d_fazenda.assign(
            idConsultor=d_fazenda['idConsultor'].str.split(';')
        ).explode('idConsultor')
        
        # Remover linhas onde idConsultor ficou vazio após a divisão (ex: se tinha '2;;30')
        # ou se a coluna original era NaN e virou ''
        d_fazenda = d_fazenda[d_fazenda['idConsultor'] != '']
        
        # Opcional: Converter idConsultor para tipo numérico (int) se for o caso
        d_fazenda['idConsultor'] = pd.to_numeric(d_fazenda['idConsultor'])
        print(d_fazenda.dtypes)
    else:
        print("⚠️ Nenhum registro de d_fazenda encontrado para o período especificado.")
        d_fazenda = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de d_fazenda: {str(e)}")
    d_fazenda = pd.DataFrame()       

try:
    df_produtores_12meses = (
        df_produtores_12meses
        .merge(
            d_fazenda,
            on = 'idFazenda',
            how='left'
        )
    )
    # df_produtores_12meses = df_produtores_12meses.loc[df_produtores_12meses['codigo_lr'].notna()] # Retirar os que não tem código LR
    df_produtores_12meses.drop('idFazenda', axis='columns', inplace=True) # Retirar idFazenda para não poluir a tabela
    df_produtores_12meses.rename(columns={'mesReferencia':'mes_elabore'},inplace=True)
except Exception as e:
    print(f"❌ Erro ao mesclar produtores com dados: {str(e)}")
    df_produtores_12meses = pd.DataFrame()   

# ✅ Remover ANTES do merge com df_final
df_produtores_12meses = df_produtores_12meses.drop_duplicates(
    subset=['codigo_lr', 'mes_elabore'],
    keep='first'
)
print(f"✅ Duplicatas removidas de df_produtores_12meses. Restam: {len(df_produtores_12meses)}")

# Aí sim realizar o merge
df_final = df_final.merge(
    df_produtores_12meses[['codigo_lr', 'mes_elabore', 'meses_sequenciais', 'idConsultor']],
    on=['codigo_lr', 'mes_elabore'],
    how='left'
)
display(df_final.head())

#  ETAPA 8: Inserindo tabela de PRODUTORES 12 MESES
print("\n⬆️ ETAPA 8: INSERINDO A PROFISSÃO DO CONSULTOR")


def padronizar_nome_consultor(df, coluna_nome='nome_consultor'):
    """
    Aplica padronização (maiúsculas, sem acentos, sem espaços extras)
    na coluna nome_consultor de um DataFrame.
    """
    if coluna_nome in df.columns:
        print(f"🔄 Padronizando coluna '{coluna_nome}'...")
        df[coluna_nome] = df[coluna_nome].astype(str).apply(
            lambda x: unidecode(x).upper().strip() if pd.notna(x) else x
        )
        print(f"✅ Coluna '{coluna_nome}' padronizada.")
    else:
        print(f"⚠️ Coluna '{coluna_nome}' não encontrada no DataFrame.")
    return df

# Importar tab_fazenda para identificar codigo_lr por idFazenda
try:
    tab_consultor = supabase.table('tab_consultor') \
        .select('formacaoConsultor,nomeConsultor') \
        .eq('Excluido', 0) \
        .execute()
    if tab_consultor.data:
        d_consultor = pd.DataFrame(tab_consultor.data)
        # PRIMEIRO renomear
        d_consultor.columns = ['profissao_consultor', 'nome_consultor']
        # DEPOIS criar o merge
        d_consultor_merge = d_consultor[['nome_consultor', 'profissao_consultor']].copy()
        # Aplicar a mesma lógica de LAC CONSULTORIA e padronização
        consultores_lac = ['CELIO ROBERTO OLIVEIRA', 'SUELY DE JESUS OLIVEIRA']
        d_consultor['nome_consultor'] = d_consultor['nome_consultor'].apply(
            lambda nome: 'LAC CONSULTORIA' if nome in consultores_lac else nome
        )
        d_consultor = padronizar_nome_consultor(d_consultor, 'nome_consultor')
        # Remover duplicatas se houver, para garantir um merge 1:1 ou N:1
        d_consultor = d_consultor.drop_duplicates(subset=['nome_consultor'], keep='first')
        print(d_consultor_merge.dtypes)
        print(d_consultor_merge.head())

    else:
        print("⚠️ Nenhum registro de consultor encontrado para o período especificado.")
        d_consultor = pd.DataFrame()
except Exception as e:
    print(f"❌ Erro ao importar dados de consultores: {str(e)}")
    d_consultor = pd.DataFrame()
    
df_final = padronizar_nome_consultor(df_final, 'nome_consultor')
df_final = df_final.drop_duplicates(
    subset=['codigo_lr', 'mes_elabore'],
    keep='first'
)
# Adicionar profissao_consultor no df_final.
try:
    # Remover a coluna idConsultor de df_final antes do merge, pois ela é a problemática
    if 'idConsultor' in df_final.columns:
        df_final.drop(columns=['idConsultor'], inplace=True)
        print("✅ Coluna 'idConsultor' removida de df_final antes do merge de profissão.")

    df_final = (
        df_final
        .merge(
            d_consultor[['nome_consultor', 'profissao_consultor']], # Selecionar apenas as colunas necessárias
            on=['nome_consultor'],
            how='left'
        )
    )
    print(f"✅ Profissão do consultor mesclada com sucesso. Total de registros: {len(df_final)}")
    display(df_final.head())
    
except Exception as e:
    print(f"❌ Erro ao mesclar dados de consultores: {str(e)}")

#  NOVA ETAPA 9: Preparar df_final para UPSERT no Supabase
print("\n⬆️ ETAPA 9: PREPARANDO E REALIZANDO UPSERT NO SUPABASE")

# Adicionar a coluna data_processamento
df_final['data_processamento'] = datetime.now(timezone.utc)

# Data limite
data_limite = datetime(hoje.year, hoje.month, 1,0,0,0)

# Limitar o número de linhas
df_final = df_final.loc[df_final['mes_referencia']<=data_limite]

#  CORREÇÃO AQUI: Converter todas as colunas de data/hora para string ISO 8601
# Identificar colunas que são de data/hora
date_cols = ['mes_referencia', 'data_carencia_fim', 'mes_elabore', 'data_processamento', 'data_inicio_vinculo', 'data_referencia']

for col in date_cols:
    if col in df_final.columns:
        df_final[col] = pd.to_datetime(df_final[col], errors='coerce')

        if df_final[col].dt.tz is not None:
            df_final[col] = df_final[col].dt.tz_convert('UTC').dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ')
        else:
            if col in ['mes_referencia', 'data_carencia_fim', 'mes_elabore', 'data_inicio_vinculo', 'data_referencia']:
                df_final[col] = df_final[col].dt.strftime('%Y-%m-%d')
            else:
                df_final[col] = df_final[col].dt.strftime('%Y-%m-%dT%H:%M:%S')

        df_final[col] = df_final[col].replace({pd.NaT: None}) # Já trata NaT para None

# ✅ NOVO BLOCO: TRATAMENTO FINAL DE np.nan PARA None E CONVERSÃO DE TIPOS
print("\n🔄 Tratando np.nan para None e ajustando tipos de colunas numéricas...")

# Colunas que devem ser inteiros, mas podem ter nulos
int_nullable_cols = ['excecao', 'meses_sequenciais']
for col in int_nullable_cols:
    if col in df_final.columns:
        # Primeiro, garantir que np.nan seja tratado para None
        df_final.loc[df_final[col].isna(), col] = None
        # Em seguida, converter para o tipo inteiro que aceita nulos (Int64)
        # Isso também converterá None para o valor nulo do Int64
        try:
            df_final[col] = df_final[col].astype('Int64')
            print(f"  ✅ Coluna '{col}': np.nan transformados para None e tipo ajustado para Int64.")
        except Exception as e:
            print(f"  ❌ Erro ao converter coluna '{col}' para Int64: {e}. Mantendo tipo original.")

# Para outras colunas que podem ter np.nan e não são datas (já tratadas) ou Int64
for col in df_final.columns:
    # Se a coluna não é uma das datas ou Int64 já tratadas, e não é string
    if col not in date_cols and col not in int_nullable_cols and not pd.api.types.is_object_dtype(df_final[col]):
        if df_final[col].isnull().any():
            df_final.loc[df_final[col].isnull(), col] = None
            print(f"  ✅ Coluna '{col}': np.nan transformados para None.")
    # Para colunas de objeto (string) que podem ter a string 'NaN' ou np.nan
    elif pd.api.types.is_object_dtype(df_final[col]):
        if (df_final[col] == 'NaN').any():
            df_final.loc[df_final[col] == 'NaN', col] = None
            print(f"  ✅ Coluna '{col}': string 'NaN' transformadas para None.")
        if df_final[col].isnull().any():
            df_final.loc[df_final[col].isnull(), col] = None
            print(f"  ✅ Coluna '{col}': np.nan em objeto transformados para None.")


# Separar apenas as colunas necessárias
tab_supabase_cols = [
    'codigo_lr', 'nome_consultor', 'profissao_consultor', 'projeto',
    'mes_referencia', 'data_carencia_fim', 'mes_elabore',
    'consistencia_mensal', 'consistencia_anual', 'status_code', 'excecao', 'meses_sequenciais',
    'detalhamento_inconsistencia',
    'data_processamento'
]
# Garantir que só seleciona colunas que existem no df_final
tab_supabase_cols = [c for c in tab_supabase_cols if c in df_final.columns]
# Slice dataframe
df_final = df_final[tab_supabase_cols]

# ❌ REMOVER ESTA LINHA: df_final.loc[df_final['meses_sequenciais']=='NaN','meses_sequenciais'] = None
# O novo bloco acima já trata isso de forma mais robusta.

# Exibir o head novamente para verificar os tipos após a conversão
print("\nDataFrame df_final (head após conversão de datas para string e tratamento de NaN):")
print(df_final.head())
print("\nTipos de dados de df_final após conversão de datas e tratamento de NaN:")
print(df_final.dtypes)


# Exemplo de como seria o upsert no seu script Python
SUPABASE_TABLE_CONSISTENCIA = TAB_CONSISTENCIA_FATO

records_to_upsert = df_final.to_dict(orient='records')

chunk_size = 1000
for i in range(0, len(records_to_upsert), chunk_size):
    chunk = records_to_upsert[i:i + chunk_size]
    try:
        response = supabase.table(SUPABASE_TABLE_CONSISTENCIA).upsert(
            chunk,
            on_conflict="codigo_lr,nome_consultor,mes_referencia" # Chave composta para o UPSERT
        ).execute()
        if response.data:
            print(f"     - Upsert de {len(response.data)} registros em chunk {i//chunk_size + 1}.")
        else:
            print(f"     - Nenhum registro upserted para chunk {i//chunk_size + 1} ou erro na resposta.")
    except Exception as e:
        print(f"❌ Erro ao realizar upsert no Supabase (chunk {i//chunk_size + 1}): {str(e)}")


print("\nDataFrame df_final (head após merges de consistência):")
print(df_final.head())

# 9. Tabela Fato: Movimentação de Produtores (f_mov_produtores)

In [ ]:
#  NOVA ETAPA 1.5: Importar dados de solicitação de vínculo (tab_vinculos_sq_backup)
print("\n🔍 ETAPA 1: IMPORTANDO DATAS DE SOLICITAÇÃO DE VÍNCULO (tab_vinculos_sq_backup)")
colunas_vinculos_sq = '*'

# Projetos alvo
projetos = ['REGENERA', 'ALVOAR ECO', 'ALVOAR ASSIST', 'SEMEAR', 'ATEG_CCPR', 'Alvoar', 'LPA']

# Consultores CFT (Sair)
grupo_cft = [
    'DAYANNE UCHOA VEIGA / DEBORA LIMA DE OLIVEIRA / MARIO BARBOSA ROSA FILHO / MATEUS CARNIELLI / TALITA FONTES / THAYNAN FERREIRA DE ARAUJO',
    'HUGO LOPES / MATEUS CARNIELLI / ROMARCIO PAULO DE OLIVEIRA / THAYNAN FERREIRA DE ARAUJO',
    'BRUNO ANTONIO FERRONI RODRIGUES / HUGO LOPES / MATEUS CARNIELLI / THAYNAN FERREIRA DE ARAUJO',
    'MATHEUS GOMIDES GONCALVES', 'TALITA FONTES'
]

try:
    # Importar todos os registros, sem filtro de data, para garantir que todas as datas de solicitação sejam capturadas
    df_vinculos_solicitacao_supabase = (
        supabase
        .table(TAB_VINCULOS_STAGING)
        .select(colunas_vinculos_sq)
        .in_('projeto', projetos)
        .execute()
    )
    
    if df_vinculos_solicitacao_supabase.data:
        df_vinculos = pd.DataFrame(df_vinculos_solicitacao_supabase.data)
        df_vinculos.rename(columns={'consultor_grupo_atendimento': 'nome_consultor', 'data_associacao':'data_movimentacao'}, inplace=True)
        df_vinculos = df_vinculos.loc[~df_vinculos['nome_consultor'].isin(grupo_cft)]
        # Após essa linha em AMBAS as etapas (df_vinculos e df_inativacao):
        df_vinculos['data_movimentacao'] = pd.to_datetime(df_vinculos['data_movimentacao']).dt.to_period('M').dt.to_timestamp()
        # Adicione imediatamente abaixo:
        nulos_data = df_vinculos['data_movimentacao'].isna().sum()
        if nulos_data > 0:
            print(f"⚠️ {nulos_data} registros com data_movimentacao nula — serão removidos")
            df_vinculos = df_vinculos.dropna(subset=['data_movimentacao'])
        # Faça o mesmo para df_inativacao
        df_vinculos['movimentacao'] = 'Entrada'
        df_vinculos['motivo_inativacao'] = None
        df_vinculos['outro_motivo'] = None

        print("⚠️ Nenhum registro de solicitação de vínculo encontrado.")

except Exception as e:
    print(f"❌ Erro ao importar dados de solicitação de vínculo: {str(e)}")

#  NOVA ETAPA 2: Importar dados de solicitação de inativação (tab_inativacao_sq)
print("\n🔍 ETAPA 2: IMPORTANDO DATAS DE SOLICITAÇÃO DE VÍNCULO (tab_vinculos_sq_backup)")
colunas_vinculos_sq = 'codigo_lr,nome_consultor,data_solicitacao,motivo_inativacao,outro_motivo'
try:
    # Importar todos os registros, sem filtro de data, para garantir que todas as datas de solicitação sejam capturadas
    df_inativacao_supabase = (
        supabase
        .table(TAB_INATIVACAO_PRODUTORES)
        .select(colunas_vinculos_sq)
        .in_('projeto', projetos)
        .execute()
    )
    
    if df_inativacao_supabase.data:
        df_inativacao = pd.DataFrame(df_inativacao_supabase.data)
        df_inativacao.rename(columns={'consultor_grupo_atendimento': 'nome_consultor', 'data_solicitacao':'data_movimentacao'}, inplace=True)
        df_inativacao = df_inativacao.loc[~df_inativacao['nome_consultor'].isin(grupo_cft)]
        df_inativacao['data_movimentacao'] = pd.to_datetime(df_inativacao['data_movimentacao']).dt.to_period('M').dt.to_timestamp()
        nulos = df_inativacao['data_movimentacao'].isna().sum()
        if nulos > 0:
            print(f"⚠️ {nulos} registros removidos por data_movimentacao nula (df_inativacao)")
            df_inativacao = df_inativacao.dropna(subset=['data_movimentacao'])
        df_inativacao['movimentacao'] = 'Saída'

        print("⚠️ Nenhum registro de solicitação de vínculo encontrado.")
        
except Exception as e:
    print(f"❌ Erro ao importar dados de solicitação de vínculo: {str(e)}")


#  NOVA ETAPA 3: CRIAR ID_COMPOSTO
print("\n🔍 ETAPA 3: CRIAR ID_COMPOSTO")
df_movimentacao = pd.concat([df_vinculos, df_inativacao])

# Usamos .dt.strftime('%Y-%m-%d') para a data para um formato consistente.
df_movimentacao['id_composto'] = (
    df_movimentacao['codigo_lr'].astype(str) + '_' +
    df_movimentacao['nome_consultor'].astype(str) + '_' +
    df_movimentacao['data_movimentacao'].dt.strftime('%Y-%m-%d').astype(str) + '_' +
    df_movimentacao['movimentacao'].astype(str)
)
print(f"✅ Coluna 'id_composto' criada. Exemplo: {df_movimentacao['id_composto'].iloc[0]}")
print("\nDataFrame df_movimentacao (head com id_composto):")
print(df_movimentacao.head())


#  ETAPA 4: Obter id_composto existentes do Supabase
print("\n🔍 ETAPA 4: IMPORTANDO ID COMPOSTO EXISTENTES DO SUPABASE")
SUPABASE_TABLE_MOVIMENTACAO = TAB_MOVIMENTACAO_FATO

try:
    # Selecionar apenas a coluna id_composto do Supabase
    response_supabase_ids = supabase.table(SUPABASE_TABLE_MOVIMENTACAO).select('id_composto').execute()
    if response_supabase_ids.data:
        df_ids_supabase = pd.DataFrame(response_supabase_ids.data)
        # Converter a coluna para um set para buscas mais rápidas
        existing_ids_supabase = set(df_ids_supabase['id_composto'].tolist())
        print(f"✅ {len(existing_ids_supabase)} IDs compostos existentes importados do Supabase.")
        print("⚠️ Nenhuma ID composta encontrada no Supabase. Todos os registros serão considerados novos.")
        existing_ids_supabase = set()
except Exception as e:
    print(f"❌ Erro ao importar IDs compostos do Supabase: {str(e)}")
    existing_ids_supabase = set() # Em caso de erro, assume que não há IDs existentes

#  ETAPA 5: Filtrar o DataFrame local para manter apenas as linhas novas
print("\n⚙️ ETAPA 5: FILTRANDO NOVOS REGISTROS")

# Filtrar df_movimentacao para manter apenas os IDs que não estão no Supabase
df_novos_registros = df_movimentacao[~df_movimentacao['id_composto'].isin(existing_ids_supabase)].copy()

# Remover duplicados
if not df_novos_registros.empty:
    num_duplicatas = df_novos_registros.duplicated(subset=['id_composto']).sum()
    if num_duplicatas > 0:
        print(f"⚠️ {num_duplicatas} duplicatas de 'id_composto' encontradas no DataFrame de novos registros. Removendo...")
        df_novos_registros.drop_duplicates(subset=['id_composto'], keep='first', inplace=True)
        print(f"✅ Duplicatas removidas. Restam {len(df_novos_registros)} registros únicos.")
        print("✅ Nenhuma duplicata de 'id_composto' encontrada no DataFrame de novos registros.")

if not df_novos_registros.empty:
    print(f"✅ {len(df_novos_registros)} novos registros identificados para inserção.")
    print("\nDataFrame df_novos_registros (head):")
    print(df_novos_registros.head())
    print("⚠️ Nenhum novo registro encontrado. O Supabase já está atualizado.")

#  ETAPA 6: Realizar o upsert/insert no Supabase
print("\n⬆️ ETAPA 6: REALIZANDO UPSERT DE NOVOS REGISTROS NO SUPABASE")

if not df_novos_registros.empty:
    # Preparar os dados para upsert
    # Garantir que colunas de data/hora sejam strings ISO 8601 e NaN/NaT sejam None
    # (Adapte este bloco de tratamento de datas/NaN conforme as colunas do seu df_movimentacao)

    # Exemplo de tratamento para 'data_movimentacao' e outras colunas que podem ter NaN
    df_novos_registros['data_movimentacao'] = df_novos_registros['data_movimentacao'].dt.strftime('%Y-%m-%d').replace({pd.NaT: None})

    # Tratar outras colunas que podem ter NaN (como 'motivo_inativacao', 'outro_motivo')
    for col in ['motivo_inativacao', 'outro_motivo']:
        if col in df_novos_registros.columns:
            df_novos_registros.loc[df_novos_registros[col].isna(), col] = None

    # Selecionar apenas as colunas que existem na tabela do Supabase
    cols_movimentacao_validas = ['id_composto', 'codigo_lr', 'nome_consultor', 'data_movimentacao', 'movimentacao', 'motivo_inativacao', 'outro_motivo', 'data_processamento']
    cols_presentes = [c for c in cols_movimentacao_validas if c in df_novos_registros.columns]
    df_para_upsert = df_novos_registros[cols_presentes].copy()

    # Converter todos os NaN/NaT float para None (evita JSON serialization error)
    import numpy as np
    for col in df_para_upsert.columns:
        if df_para_upsert[col].dtype == object:
            df_para_upsert[col] = df_para_upsert[col].where(df_para_upsert[col].notna(), None)
        elif df_para_upsert[col].dtype in [float, 'float64']:
            df_para_upsert[col] = df_para_upsert[col].apply(lambda x: None if (pd.isna(x) or (isinstance(x, float) and (x != x))) else x)

    records_to_upsert = df_para_upsert.to_dict(orient='records')

    chunk_size = 1000
    for i in range(0, len(records_to_upsert), chunk_size):
        chunk = records_to_upsert[i:i + chunk_size]
        try:
            # Para novos registros, um 'insert' simples pode ser suficiente se a chave composta
            # for garantidamente única e você não quiser atualizar registros existentes.
            # Se você quiser que ele atualize se encontrar a chave composta, use 'upsert'.
            # A chave 'on_conflict' deve ser a coluna 'id_composto' no Supabase.
            
            # Descomente para gravação oficial:
            response = supabase.table(SUPABASE_TABLE_MOVIMENTACAO).upsert(chunk, on_conflict="id_composto").execute()
            if response and hasattr(response, 'data') and response.data:
                print(f"     - Upsert de {len(response.data)} registros em chunk {i//chunk_size + 1}.")
                print(f"     - Chunk {i//chunk_size + 1} enviado com sucesso.")

        except Exception as e:
            print(f"❌ Erro ao realizar upsert no Supabase (chunk {i//chunk_size + 1}): {str(e)}")
    print("✅ Nenhuma inserção necessária.")

print("\nProcesso de atualização de movimentação concluído.")

In [ ]:
# ùltinmo